In [2]:
from __future__ import annotations

import json
import hashlib
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import Lasso, ElasticNet, Ridge, LassoLars
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GroupKFold, ParameterGrid, StratifiedKFold
from joblib import Parallel, delayed
import os
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import f_regression

# Optional XGBoost candidate.
# If xgboost is not installed in the current environment, the xgboost model
# will be removed from MODEL_TYPES below to avoid crashing the whole run.
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBRegressor = None
    XGBOOST_AVAILABLE = False


In [3]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [4]:
# ==========================================
# Config
# ==========================================
DATA_ROOT = Path("../../DifferentCom_data_rebuild")
TP_DIR = DATA_ROOT / "tp_views"
BASE_DIR = DATA_ROOT / "base_patient_tables"

# ==========================================
# Experiment tags / output directories
# ==========================================
# IMPORTANT:
# - TRAIN_EXPERIMENT_TAG must match the completed training-only OOF/CV notebook.
# - TEST_EXPERIMENT_TAG is only for this held-out test run.
# - The holdout test set is NOT used to choose model/configs.
TRAIN_EXPERIMENT_TAG = "single_omics_train_paper_style_v11_rescue_expansion_v9compatible"
TEST_EXPERIMENT_TAG = "single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible"

# Compatibility alias for older helper functions in this notebook.
EXPERIMENT_TAG = TEST_EXPERIMENT_TAG

TRAIN_SAVE_DIR = DATA_ROOT / f"results_single_omics_train_only_{TRAIN_EXPERIMENT_TAG}"
TRAIN_SUMMARY_PATH = TRAIN_SAVE_DIR / f"omics_paper_style_train_summary_{TRAIN_EXPERIMENT_TAG}.csv"

SAVE_DIR = DATA_ROOT / "testResult" / f"results_single_omics_test_{TEST_EXPERIMENT_TAG}"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PATH = BASE_DIR / "target_by_patient.csv"

# A = proteomics, B = metabolomics, C = miRNA
COMBOS = ["A", "B", "C"]
TISSUES = ["csf", "ser"]
TIMEPOINTS = [24, 48, 72, 96, 120]

# Must match the training v11 available model families.
# The final held-out test cell still restricts each job to the model selected by training OOF/CV.
MODEL_TYPES = [
    "ridge",
    "elasticnet",
    "pls",
    "svr_linear",
    "gbr",
    "xgboost",
]

# Keep the notebook runnable even when xgboost is not installed.
if not XGBOOST_AVAILABLE:
    print("[WARN] xgboost is not installed. Removing 'xgboost' from MODEL_TYPES.")
    MODEL_TYPES = [m for m in MODEL_TYPES if m != "xgboost"]

# Feature-selection modes available in v11 training.
FEATURE_SELECTION_MODES_TO_RUN = [
    "f_regression_topk",
    "corr_topk",
]

# v11 training used light clinical mode only.
CLINICAL_MODES_TO_RUN = ["light"]

# sparse filter: train 기준으로만 적용
MIN_OBS_FRAC = 0.50

# CV used inside train-only model fitting on TRAIN_IDS.
N_SPLITS_OUTER = 5
N_SPLITS_INNER = 3
RANDOM_STATE = 42

# y-bin stratification for regression CV stability
USE_STRATIFIED_OUTER_CV = True
N_Y_BINS_FOR_OUTER_CV = 3
OUTER_CV_RANDOM_STATE = 42

# Must match training v11 model-selection penalty.
INNER_SELECTION_STD_PENALTY = 0.10

# ==========================================
# Held-out test config selection policy
# ==========================================
# Safe final-test default:
#   Choose configs using TRAINING OOF/CV only, then evaluate those pre-selected configs once on TEST_IDS.
#
# Recommended final EF/LF comparison:
#   "best_per_tissue_tp"         -> 2 tissues x 5 TPs = up to 10 rows.
#
# Recommended diagnostic after unstable holdout results:
#   "top_n_per_tissue_tp"        -> top N training-OOF configs per tissue+TP.
#                                   This checks whether the OOF-best config is unstable on holdout.
#                                   Do NOT use holdout test R² to choose the final config.
#
# Other optional diagnostics:
#   "best_per_tissue_combo_tp"   -> 2 tissues x 3 combos x 5 TPs = up to 30 rows.
#   "top_k_per_tissue_combo_tp"  -> top K per tissue/combo/tp from training summary.
#   "all_train_ok"              -> diagnostic only; do not use to select final best by test R².
TEST_CONFIG_SELECTION_MODE = "top_n_per_tissue_tp"
TOP_N_PER_TISSUE_TP = 5
TOP_K_PER_TISSUE_COMBO_TP = 3

# Training-CV selection ranking columns.
# test_r2 must NEVER be used here.
TRAIN_SELECTION_SORT_COLUMNS = [
    "oof_r2",
    "mean_valid_r2",
    "final_full_train_best_score_inner_oof_r2",
    "oof_mae",
]

# ==========================================
# Leakage-free feature-wise weighting
# ==========================================
# Note:
# - feature weighting is computed only after fold-safe feature selection.
# - inner CV: inner-train only
# - final test: predefined train only
USE_FEATURE_WEIGHTING = True
FEATURE_WEIGHT_SCORE_MODE = "univariate_corr_soft_conservative"
FEATURE_WEIGHT_MIN = 1.0
FEATURE_WEIGHT_MAX = 1.20
FEATURE_WEIGHT_MIN_OBS = 8
FEATURE_WEIGHT_EXCLUDE_CLINICAL = True
FEATURE_WEIGHT_DEFAULT = 1.0

FEATURE_TRACE_DIR = SAVE_DIR / "feature_traces"
FEATURE_TRACE_DIR.mkdir(parents=True, exist_ok=True)

DETAIL_DIR = SAVE_DIR / "single_detail_outputs"
DETAIL_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_SUMMARY_PATH:", TRAIN_SUMMARY_PATH.resolve())
print("SAVE_DIR:", SAVE_DIR.resolve())
print("DETAIL_DIR:", DETAIL_DIR.resolve())
print("TRAIN_EXPERIMENT_TAG:", TRAIN_EXPERIMENT_TAG)
print("TEST_EXPERIMENT_TAG:", TEST_EXPERIMENT_TAG)
print("TEST_CONFIG_SELECTION_MODE:", TEST_CONFIG_SELECTION_MODE)
print("MODEL_TYPES available:", MODEL_TYPES)
print("FEATURE_SELECTION_MODES_TO_RUN:", FEATURE_SELECTION_MODES_TO_RUN)
print("CLINICAL_MODES_TO_RUN:", CLINICAL_MODES_TO_RUN)
print("USE_FEATURE_WEIGHTING:", USE_FEATURE_WEIGHTING)
print("FEATURE_WEIGHT_SCORE_MODE:", FEATURE_WEIGHT_SCORE_MODE)
print("FEATURE_WEIGHT_RANGE:", (FEATURE_WEIGHT_MIN, FEATURE_WEIGHT_MAX))


TRAIN_SUMMARY_PATH: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/results_single_omics_train_only_single_omics_train_paper_style_v11_rescue_expansion_v9compatible/omics_paper_style_train_summary_single_omics_train_paper_style_v11_rescue_expansion_v9compatible.csv
SAVE_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/testResult/results_single_omics_test_single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible
DETAIL_DIR: /data2/jinoh001/stable/research_SPI/DifferentCom_data_rebuild/testResult/results_single_omics_test_single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible/single_detail_outputs
TRAIN_EXPERIMENT_TAG: single_omics_train_paper_style_v11_rescue_expansion_v9compatible
TEST_EXPERIMENT_TAG: single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible
TEST_CONFIG_SELECTION_MODE: top_n_per_tissue_tp
MODEL_TYPES available: ['ridge', 'elasticnet', 'pls', 'svr_linear', 'gbr', 'xgboos

In [5]:
# ==========================================
# Parallel config
# ==========================================
TOTAL_CPUS = os.cpu_count() or 1

# 실험 단위 병렬화 권장
# 32 CPU 기준이면 6~8 정도부터 시작하는 게 안전
N_JOBS_EXPERIMENT = 8

print("TOTAL_CPUS:", TOTAL_CPUS)
print("N_JOBS_EXPERIMENT:", N_JOBS_EXPERIMENT)

TOTAL_CPUS: 32
N_JOBS_EXPERIMENT: 8


In [6]:
def read_id_txt(path: str) -> list[str]:
    """Read a one-column patient-id file.

    Expected format:
        Patient
        <id1>
        <id2>
        ...

    The function keeps IDs as strings so they match feature-table indexes.
    """
    df = pd.read_csv(path)
    if "Patient" not in df.columns:
        raise ValueError(f"{path} must contain a 'Patient' column. Found: {df.columns.tolist()}")
    ids = df["Patient"].dropna().astype(str).tolist()
    # keep order while removing duplicates
    return list(dict.fromkeys(ids))


TRAIN_ID_PATH = "../../data/training_id.txt"
TEST_ID_PATH = "../../data/testing_id.txt"

TRAIN_IDS = read_id_txt(TRAIN_ID_PATH)
TEST_IDS = read_id_txt(TEST_ID_PATH)

overlap_ids = sorted(set(TRAIN_IDS).intersection(TEST_IDS))

print("n TRAIN_IDS:", len(TRAIN_IDS))
print("n TEST_IDS :", len(TEST_IDS))
print("n overlap  :", len(overlap_ids))
print("overlap examples:", overlap_ids[:10])

if len(TEST_IDS) == 0:
    raise ValueError("TEST_IDS is empty. Check TEST_ID_PATH.")
if len(overlap_ids) > 0:
    raise ValueError("TRAIN_IDS and TEST_IDS overlap. This would leak holdout-test data.")


n TRAIN_IDS: 60
n TEST_IDS : 30
n overlap  : 0
overlap examples: []


In [7]:
def load_target(path: Path) -> pd.Series:
    """
    patient-level target loader
    """
    y = pd.read_csv(path, index_col=0).iloc[:, 0]
    y.index = y.index.astype(str)
    y.name = "DeltaTMS"
    return y


In [8]:
# ==========================================
# Alignment helpers
# ==========================================
def align_xy(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """
    Keep only patients that exist in both X and y.
    """
    idx = X.index.intersection(y.index)
    X = X.loc[idx].copy()
    y = y.loc[idx].copy()
    return X, y

In [9]:
def filter_sparse_features_train_test(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    min_obs_frac: float = 0.7,
    protected_keep_cols: list[str] | None = None,
):
    if protected_keep_cols is None:
        protected_keep_cols = []

    protected_keep_cols = [c for c in protected_keep_cols if c in X_train.columns]

    # 1) constant columns 제거 (train 기준)
    nunique = X_train.nunique(dropna=True)
    const_drop_cols = [
        c for c in X_train.columns
        if nunique.get(c, 0) <= 1 and c not in protected_keep_cols
    ]
    X_train_1 = X_train.drop(columns=const_drop_cols)
    X_valid_1 = X_valid.drop(columns=[c for c in const_drop_cols if c in X_valid.columns])

    # 2) missing ratio 기준 filtering (train 기준)
    obs_frac = X_train_1.notna().mean(axis=0)
    keep_cols = [
        c for c in X_train_1.columns
        if (obs_frac.get(c, 0.0) >= min_obs_frac) or (c in protected_keep_cols)
    ]

    X_train_f = X_train_1.loc[:, keep_cols].copy()
    X_valid_f = X_valid_1.reindex(columns=keep_cols).copy()

    return X_train_f, X_valid_f

In [10]:
def _median_impute_numeric_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fit median imputation on X_train only, then apply to X_train/X_valid.
    This helper is used only for univariate scoring, not for final model fitting.
    """
    X_tr = X_train.apply(pd.to_numeric, errors="coerce")
    X_va = X_valid.apply(pd.to_numeric, errors="coerce")

    med = X_tr.median(axis=0, skipna=True)
    med = med.fillna(0.0)

    return X_tr.fillna(med), X_va.fillna(med)


def compute_univariate_feature_scores(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    protected_keep_cols: list[str] | None = None,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
) -> pd.DataFrame:
    """
    Compute train-only univariate scores.

    Available score columns:
    - abs_corr: absolute Pearson correlation with DeltaTMS
    - f_score / p_value: sklearn f_regression score and p-value
    - neg_log10_p: larger means smaller p-value
    - hybrid_rank_score: combines corr-rank and p-value-rank

    Important:
    - This function must be called only on the training side of the current split.
    - It should never see validation/test y.
    """
    protected_keep_cols = protected_keep_cols or []
    protected_set = set(protected_keep_cols)

    candidate_cols = [c for c in X_train.columns if c not in protected_set]
    rows = []

    if len(candidate_cols) == 0:
        return pd.DataFrame(columns=[
            "feature", "abs_corr", "f_score", "p_value", "neg_log10_p",
            "corr_rank_score", "p_rank_score", "hybrid_rank_score",
            "is_protected",
        ])

    X_cand = X_train.loc[:, candidate_cols].copy()
    X_imp, _ = _median_impute_numeric_train_valid(X_cand, X_cand)
    y_num = pd.to_numeric(y_train, errors="coerce")

    valid_y = y_num.notna() & np.isfinite(y_num)
    X_imp = X_imp.loc[valid_y]
    y_num = y_num.loc[valid_y]

    if len(y_num) >= 3 and y_num.nunique(dropna=True) > 1:
        try:
            f_vals, p_vals = f_regression(X_imp, y_num)
        except Exception:
            f_vals = np.zeros(len(candidate_cols), dtype=float)
            p_vals = np.ones(len(candidate_cols), dtype=float)
    else:
        f_vals = np.zeros(len(candidate_cols), dtype=float)
        p_vals = np.ones(len(candidate_cols), dtype=float)

    for j, col in enumerate(candidate_cols):
        abs_corr = _safe_abs_corr(X_train[col], y_train, min_obs=min_obs)
        f_score = float(f_vals[j]) if np.isfinite(f_vals[j]) else 0.0
        p_value = float(p_vals[j]) if np.isfinite(p_vals[j]) else 1.0
        p_value = max(p_value, 1e-300)
        rows.append({
            "feature": col,
            "abs_corr": float(abs_corr),
            "f_score": f_score,
            "p_value": p_value,
            "neg_log10_p": float(-np.log10(p_value)),
            "is_protected": False,
        })

    score_df = pd.DataFrame(rows)

    # rank scores: higher is better
    if len(score_df) > 0:
        score_df["corr_rank_score"] = score_df["abs_corr"].rank(method="average", pct=True)
        score_df["p_rank_score"] = score_df["neg_log10_p"].rank(method="average", pct=True)
        score_df["hybrid_rank_score"] = 0.5 * score_df["corr_rank_score"] + 0.5 * score_df["p_rank_score"]

    return score_df


def select_features_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_train: pd.Series,
    k: int | None,
    mode: str = "f_regression_topk",
    protected_keep_cols: list[str] | None = None,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Fold-safe feature selection.

    This replaces the old approach where target-aware top-k was applied once
    before inner CV. Here, feature selection is repeated inside each inner fold
    using only that fold's training data.

    Modes:
    - "none": keep all columns
    - "corr_topk": top-k by abs correlation with y_train
    - "f_regression_topk": top-k by univariate f_regression p-value
    - "hybrid_topk": top-k by combined corr/p-value rank
    """
    protected_keep_cols = protected_keep_cols or []
    protected_keep_cols = [c for c in protected_keep_cols if c in X_train.columns]

    if mode == "none" or k is None or k <= 0:
        selected_features = list(X_train.columns)
        score_df = pd.DataFrame({
            "feature": selected_features,
            "selected": True,
            "selection_mode": mode,
            "selection_score": np.nan,
            "is_protected": [c in protected_keep_cols for c in selected_features],
        })
        return (
            X_train.loc[:, selected_features].copy(),
            X_valid.reindex(columns=selected_features).copy(),
            score_df,
            selected_features,
        )

    score_df = compute_univariate_feature_scores(
        X_train=X_train,
        y_train=y_train,
        protected_keep_cols=protected_keep_cols,
        min_obs=min_obs,
    )

    if mode == "corr_topk":
        sort_col = "abs_corr"
    elif mode == "f_regression_topk":
        sort_col = "neg_log10_p"
    elif mode == "hybrid_topk":
        sort_col = "hybrid_rank_score"
    else:
        raise ValueError(f"Unknown feature_selection_mode: {mode}")

    candidate_cols = [c for c in X_train.columns if c not in set(protected_keep_cols)]
    k_for_candidates = max(0, int(k) - len(protected_keep_cols))

    if len(candidate_cols) == 0 or k_for_candidates == 0 or len(score_df) == 0:
        top_candidate_cols = []
    else:
        top_candidate_cols = (
            score_df.sort_values(sort_col, ascending=False)
                    .head(min(k_for_candidates, len(score_df)))["feature"]
                    .tolist()
        )

    selected_set = set(protected_keep_cols + top_candidate_cols)
    selected_features = [c for c in X_train.columns if c in selected_set]

    if len(selected_features) == 0:
        # Absolute fallback to avoid failed runs.
        selected_features = list(X_train.columns)

    out_score = score_df.copy()
    if len(out_score) > 0:
        out_score["selected"] = out_score["feature"].isin(selected_features)
        out_score["selection_mode"] = mode
        out_score["selection_score"] = out_score[sort_col]
        protected_rows = pd.DataFrame({
            "feature": protected_keep_cols,
            "abs_corr": np.nan,
            "f_score": np.nan,
            "p_value": np.nan,
            "neg_log10_p": np.nan,
            "corr_rank_score": np.nan,
            "p_rank_score": np.nan,
            "hybrid_rank_score": np.nan,
            "is_protected": True,
            "selected": True,
            "selection_mode": mode,
            "selection_score": np.nan,
        })
        out_score = pd.concat([protected_rows, out_score], ignore_index=True)

    return (
        X_train.loc[:, selected_features].copy(),
        X_valid.reindex(columns=selected_features).copy(),
        out_score,
        selected_features,
    )


def top_k_target_corr_filter(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
    y_train: pd.Series,
    k: int | None,
    protected_keep_cols: list[str] | None = None,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
):
    """
    Backward-compatible wrapper.
    New code should use select_features_train_valid(..., mode="corr_topk").
    """
    X_tr, X_va, _, _ = select_features_train_valid(
        X_train=X_train,
        X_valid=X_valid,
        y_train=y_train,
        k=k,
        mode="corr_topk",
        protected_keep_cols=protected_keep_cols,
        min_obs=min_obs,
    )
    return X_tr, X_va


In [11]:
from sklearn.impute import KNNImputer


def make_preprocess(X: pd.DataFrame) -> ColumnTransformer:
    cat = [c for c in X.columns if c in ["Gender", "Level"]]
    num = [c for c in X.columns if c not in cat]

    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore")),
            ]), cat),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

    try:
        preprocess.set_output(transform="pandas")
    except Exception:
        pass

    return preprocess


def make_model(model_type: str):
    """
    Model families are intentionally kept conservative because this dataset is small.
    Tree/boosting models use shallow settings through the parameter grid.
    """
    if model_type == "lasso":
        return Lasso(max_iter=50000, random_state=RANDOM_STATE, tol=1e-3)
    elif model_type == "elasticnet":
        return ElasticNet(max_iter=50000, random_state=RANDOM_STATE, tol=1e-3)
    elif model_type == "ridge":
        return Ridge()
    elif model_type == "lassolars":
        return LassoLars()
    elif model_type == "pls":
        return PLSRegression(scale=False)
    elif model_type == "svr_linear":
        return SVR(kernel="linear")
    elif model_type == "svr_rbf":
        return SVR(kernel="rbf")
    elif model_type == "knn":
        return KNeighborsRegressor()
    elif model_type == "rf":
        return RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
    elif model_type == "extratrees":
        return ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=1)
    elif model_type == "gbr":
        return GradientBoostingRegressor(random_state=RANDOM_STATE)
    elif model_type == "xgboost":
        if not XGBOOST_AVAILABLE or XGBRegressor is None:
            raise ImportError("xgboost is not installed, so model_type='xgboost' cannot be used.")
        return XGBRegressor(
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=1,
            tree_method="hist",
            verbosity=0,
        )
    elif model_type == "histgbr":
        return HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")


def make_param_grid(model_type: str) -> dict:
    """
    Reduced hyperparameter grids for the training paper-style run.

    Purpose:
    - keep the full model family list
    - reduce alpha / major model grids enough to make the run practical
    - avoid the previous very long runtime from large nested CV grids
    """
    common_num_imputers = [SimpleImputer(strategy="median")]

    if model_type == "lasso":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [0.1, 0.5],
        }

    elif model_type == "elasticnet":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [0.05, 0.1, 0.5, 1.0],
            "model__l1_ratio": [0.1, 0.2, 0.5],
        }

    elif model_type == "ridge":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [5.0, 10.0, 25.0, 50.0, 100.0],
        }

    elif model_type == "lassolars":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__alpha": [0.01, 0.1],
        }

    elif model_type == "pls":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_components": [2, 3],
        }

    elif model_type == "svr_linear":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__C": [0.03, 0.1, 0.3, 1.0],
            "model__epsilon": [0.1, 0.2],
        }

    elif model_type == "svr_rbf":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__C": [1.0],
            "model__epsilon": [0.1],
            "model__gamma": ["scale"],
        }

    elif model_type == "knn":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_neighbors": [3, 5],
            "model__weights": ["distance"],
        }

    elif model_type == "rf":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [100],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [2],
            "model__max_features": ["sqrt"],
        }

    elif model_type == "extratrees":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [100],
            "model__max_depth": [2, 3],
            "model__min_samples_leaf": [2],
            "model__max_features": ["sqrt"],
        }

    elif model_type == "gbr":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [30, 50],
            "model__learning_rate": [0.03, 0.05],
            "model__max_depth": [1],
            "model__min_samples_leaf": [3, 5],
            "model__subsample": [0.8],
        }

    elif model_type == "xgboost":
        # Conservative XGBoost grid for small-n/high-p single-omics data.
        # Goal: test regularized boosting without allowing train R2 to dominate model selection.
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__n_estimators": [50, 80],
            "model__max_depth": [1, 2],
            "model__learning_rate": [0.03],
            "model__subsample": [0.8],
            "model__colsample_bytree": [0.7, 0.9],
            "model__reg_alpha": [0.5, 1.0],
            "model__reg_lambda": [5.0, 10.0],
            "model__min_child_weight": [2, 4],
        }

    elif model_type == "histgbr":
        return {
            "preprocess__num__imputer": common_num_imputers,
            "model__max_iter": [50],
            "model__learning_rate": [0.05],
            "model__max_leaf_nodes": [5],
            "model__l2_regularization": [0.1],
        }

    else:
        raise ValueError(f"Unknown model_type: {model_type}")

def adjust_param_grid_for_training_data(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_type: str,
) -> list[dict]:
    """
    Remove parameter settings that cannot work for the current train fold.
    Example: PLS n_components must be <= min(n_samples - 1, n_features).
    """
    grid = list(ParameterGrid(make_param_grid(model_type=model_type)))

    n_samples = int(X_train.shape[0])
    n_features = int(X_train.shape[1])

    valid_grid = []
    for params in grid:
        ok = True

        if model_type == "pls":
            n_comp = int(params.get("model__n_components", 1))
            if n_comp > max(1, min(n_samples - 1, n_features)):
                ok = False

        if model_type == "knn":
            n_neighbors = int(params.get("model__n_neighbors", 5))
            if n_neighbors >= n_samples:
                ok = False

        if ok:
            valid_grid.append(params)

    if len(valid_grid) == 0:
        # Fallback to a safe default for very small folds.
        if model_type == "pls":
            return [{
                "preprocess__num__imputer": SimpleImputer(strategy="median"),
                "model__n_components": 1,
            }]
        if model_type == "knn":
            return [{
                "preprocess__num__imputer": SimpleImputer(strategy="median"),
                "model__n_neighbors": max(1, min(3, n_samples - 1)),
                "model__weights": "distance",
            }]
        return grid

    return valid_grid


def make_pipeline(
    X: pd.DataFrame,
    model_type: str,
    feature_weight_map: dict[str, float] | None = None,
) -> Pipeline:
    preprocess = make_preprocess(X)
    model = make_model(model_type)

    steps = [
        ("preprocess", preprocess),
    ]

    if USE_FEATURE_WEIGHTING:
        steps.append((
            "feature_weight",
            FeatureWeightTransformer(
                feature_weight_map=feature_weight_map,
                default_weight=FEATURE_WEIGHT_DEFAULT,
                enabled=USE_FEATURE_WEIGHTING,
            ),
        ))

    steps.append(("model", model))

    return Pipeline(steps)


def predict_1d(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    """
    Some models, e.g. PLSRegression, return shape (n, 1).
    Standardize all predictions to 1D.
    """
    return np.ravel(model.predict(X))


In [12]:
def build_feature_path(feature_mode: str, tissue: str, combo: str, tp: int) -> Path:
    if feature_mode == "omics":
        return TP_DIR / f"x_omics_{tissue}_{combo}_{tp}.csv"
    elif feature_mode == "omicsdelta":
        return TP_DIR / f"x_omicsdelta_{tissue}_{combo}_{tp}.csv"
    else:
        raise ValueError(f"Unknown feature_mode: {feature_mode}")

def build_clinical_feature_path(tissue: str, combo: str, tp: int) -> Path:
    return TP_DIR / f"x_clinical_{tissue}_{combo}_{tp}.csv"


def load_feature_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Patient" not in df.columns:
        raise ValueError(f"'Patient' not found in {path}")
    df = df.set_index("Patient")
    df.index = df.index.astype(str)
    return df

def dataframe_content_signature(df: pd.DataFrame) -> str:
    """
    Lightweight diagnostic hash used to check whether two TP feature matrices
    are accidentally identical. This is not used for modeling.
    """
    if df is None or len(df) == 0:
        return "EMPTY"
    tmp = df.copy()
    tmp = tmp.sort_index(axis=0).sort_index(axis=1)
    col_part = "|".join(map(str, tmp.columns))
    shape_part = f"{tmp.shape[0]}x{tmp.shape[1]}"
    # hash_pandas_object handles mixed dtypes after string fallback; keep it lightweight.
    try:
        values_hash = pd.util.hash_pandas_object(tmp.astype(str), index=True).values.tobytes()
    except Exception:
        values_hash = tmp.to_csv(index=True).encode("utf-8")
    return hashlib.md5((shape_part + "|" + col_part).encode("utf-8") + values_hash).hexdigest()


In [13]:
def get_clinical_columns() -> list[str]:
    return ["Age", "Gender", "Level"]


def get_hard_forbidden_proxy_columns() -> list[str]:
    return [
        "TMS_Base", "UEMS_Base", "LEMS_Base",
        "TMS_6mo", "UEMS_6mo", "LEMS_6mo",
        "AIS_6mo",
        "DeltaTMS", "DeltaUEMS", "DeltaLEMS",
    ]


def assert_no_forbidden_proxy_columns(
    X: pd.DataFrame,
    use_light_clinical: bool = False,
):
    hard_bad = [c for c in X.columns if c in get_hard_forbidden_proxy_columns()]
    if len(hard_bad) > 0:
        raise ValueError(f"Hard forbidden proxy/leakage columns found: {hard_bad}")

    all_clinical_like = [
        "Age", "Gender", "Level", "AIS", "AIS_Base", "AIS_Base_Numeric",
        "TMS_Base", "UEMS_Base", "LEMS_Base",
        "TMS_6mo", "UEMS_6mo", "LEMS_6mo", "AIS_6mo",
        "DeltaTMS", "DeltaUEMS", "DeltaLEMS",
    ]

    allowed = set(get_clinical_columns()) if use_light_clinical else set()
    soft_bad = [
        c for c in X.columns
        if (c in all_clinical_like and c not in allowed and c not in get_hard_forbidden_proxy_columns())
    ]

    if len(soft_bad) > 0:
        raise ValueError(f"Unexpected clinical columns found for this mode: {soft_bad}")

In [14]:
def load_light_clinical_block(
    tissue: str,
    combo: str,
    tp: int,
) -> pd.DataFrame:
    clin_path = build_clinical_feature_path(tissue=tissue, combo=combo, tp=tp)
    clin_df = load_feature_csv(clin_path)

    keep_cols = [c for c in get_clinical_columns() if c in clin_df.columns]
    clin_df = clin_df.loc[:, keep_cols].copy()

    missing_cols = [c for c in get_clinical_columns() if c not in clin_df.columns]
    if len(missing_cols) > 0:
        print(f"[WARN] missing light clinical columns in {clin_path.name}: {missing_cols}")

    if "Age" in clin_df.columns:
        clin_df["Age"] = pd.to_numeric(clin_df["Age"], errors="coerce")

    return clin_df


def merge_with_light_clinical(
    X_feat: pd.DataFrame,
    tissue: str,
    combo: str,
    tp: int,
) -> pd.DataFrame:
    clin_df = load_light_clinical_block(tissue=tissue, combo=combo, tp=tp)

    common_ids = X_feat.index.intersection(clin_df.index)
    X_feat2 = X_feat.loc[common_ids].copy()
    clin_df2 = clin_df.loc[common_ids].copy()

    X_merged = pd.concat([X_feat2, clin_df2], axis=1)

    dup_cols = X_merged.columns[X_merged.columns.duplicated()].tolist()
    if len(dup_cols) > 0:
        raise ValueError(f"Duplicate columns after light clinical merge: {dup_cols}")

    return X_merged


In [15]:
def encode_clinical_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Clinical block은 train 기준으로만 encoding한다.
    반환:
      - encoded X_train
      - encoded X_valid
      - encoded clinical column names
    """
    base_clin_cols = [c for c in get_clinical_columns() if c in X_train.columns]
    if len(base_clin_cols) == 0:
        return X_train, X_valid, []

    tr_clin = X_train.loc[:, base_clin_cols].copy()
    va_clin = X_valid.loc[:, [c for c in base_clin_cols if c in X_valid.columns]].copy()

    tr_other = X_train.drop(columns=base_clin_cols, errors="ignore").copy()
    va_other = X_valid.drop(columns=base_clin_cols, errors="ignore").copy()

    if "Age" in tr_clin.columns:
        tr_clin["Age"] = pd.to_numeric(tr_clin["Age"], errors="coerce")
    if "Age" in va_clin.columns:
        va_clin["Age"] = pd.to_numeric(va_clin["Age"], errors="coerce")

    cat_cols = [c for c in ["Gender", "Level"] if c in tr_clin.columns]

    if len(cat_cols) > 0:
        tr_clin_enc = pd.get_dummies(
            tr_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = pd.get_dummies(
            va_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = va_clin_enc.reindex(columns=tr_clin_enc.columns, fill_value=0.0)
    else:
        tr_clin_enc = tr_clin.copy()
        va_clin_enc = va_clin.copy()

    tr_clin_enc = tr_clin_enc.apply(pd.to_numeric, errors="coerce")
    va_clin_enc = va_clin_enc.apply(pd.to_numeric, errors="coerce")

    encoded_clin_cols = tr_clin_enc.columns.tolist()

    X_train_out = pd.concat([tr_other, tr_clin_enc], axis=1)
    X_valid_out = pd.concat([va_other, va_clin_enc], axis=1)

    return X_train_out, X_valid_out, encoded_clin_cols


In [16]:
# # ==========================================
# # Inner CV tuning
# # ==========================================
# def fit_best_model(
#     X_train: pd.DataFrame,
#     y_train: pd.Series,
#     groups_train: np.ndarray,
#     model_type: str,
#     n_splits_inner: int = 3,
# ):
#     """
#     Tune hyperparameters only on the training fold.
#     """
#     pipe = make_pipeline(X_train, model_type=model_type)
#     param_grid = make_param_grid(model_type=model_type)

#     if model_type in ["lasso", "elasticnet"] and "select__k" in param_grid:
#         preprocess = make_preprocess(X_train)
#         X_train_tx = preprocess.fit_transform(X_train, y_train)
#         n_features_after_preprocess = X_train_tx.shape[1]

#         valid_k = sorted({
#             int(k)
#             for k in param_grid["select__k"]
#             if int(k) <= n_features_after_preprocess
#         })

#         if len(valid_k) == 0:
#             valid_k = [max(1, n_features_after_preprocess)]

#         param_grid["select__k"] = valid_k

#     inner_cv = GroupKFold(n_splits=n_splits_inner)

#     gs = GridSearchCV(
#         estimator=pipe,
#         param_grid=param_grid,
#         scoring="r2",
#         cv=inner_cv,
#         n_jobs=12,
#         refit=True,
#     )
#     gs.fit(X_train, y_train, groups=groups_train)

#     return gs.best_estimator_, gs.best_params_, gs.best_score_

In [17]:
class FeatureWeightTransformer(BaseEstimator, TransformerMixin):
    """
    Apply precomputed feature-wise weights after preprocessing/scaling.

    Why after preprocessing?
    - If weights are applied before StandardScaler, StandardScaler can cancel out most
      multiplicative effects.
    - Applying after preprocessing means the model sees the weighted feature matrix.

    Leakage rule:
    - feature_weight_map must be computed outside this transformer using train data only.
    - validation/test y is never used here.
    """
    def __init__(
        self,
        feature_weight_map: dict[str, float] | None = None,
        default_weight: float = 1.0,
        enabled: bool = True,
    ):
        self.feature_weight_map = feature_weight_map
        self.default_weight = default_weight
        self.enabled = enabled

    def fit(self, X, y=None):
        if not self.enabled or self.feature_weight_map is None:
            self.weights_ = np.ones(X.shape[1], dtype=float)
            self.feature_names_ = list(getattr(X, "columns", [f"x{i}" for i in range(X.shape[1])]))
            return self

        if hasattr(X, "columns"):
            self.feature_names_ = list(X.columns)
            weights = []
            for feat in self.feature_names_:
                feat_str = str(feat)

                # exact match for numeric/transformed numeric columns
                w = self.feature_weight_map.get(feat_str, None)

                # fallback for one-hot names such as Gender_M or Level_C
                if w is None:
                    base = feat_str.split("_")[0]
                    w = self.feature_weight_map.get(base, self.default_weight)

                weights.append(float(w))

            self.weights_ = np.asarray(weights, dtype=float)
        else:
            # fallback: if sklearn cannot output pandas, keep neutral weighting
            self.feature_names_ = [f"x{i}" for i in range(X.shape[1])]
            self.weights_ = np.ones(X.shape[1], dtype=float)

        return self

    def transform(self, X):
        if not self.enabled:
            return X

        if hasattr(X, "copy") and hasattr(X, "columns"):
            Xw = X.copy()
            for col, w in zip(Xw.columns, self.weights_):
                Xw[col] = Xw[col] * float(w)
            return Xw

        return X * self.weights_


def _safe_abs_corr(x: pd.Series, y: pd.Series, min_obs: int) -> float:
    x_num = pd.to_numeric(x, errors="coerce")
    y_num = pd.to_numeric(y, errors="coerce")
    mask = x_num.notna() & y_num.notna() & np.isfinite(x_num) & np.isfinite(y_num)

    if int(mask.sum()) < min_obs:
        return 0.0

    xv = x_num.loc[mask]
    yv = y_num.loc[mask]

    if xv.nunique(dropna=True) <= 1 or yv.nunique(dropna=True) <= 1:
        return 0.0

    corr = xv.corr(yv)
    if pd.isna(corr) or not np.isfinite(corr):
        return 0.0

    return float(abs(corr))


def compute_feature_weight_map_from_train(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    protected_keep_cols: list[str] | None = None,
    min_weight: float = FEATURE_WEIGHT_MIN,
    max_weight: float = FEATURE_WEIGHT_MAX,
    min_obs: int = FEATURE_WEIGHT_MIN_OBS,
) -> tuple[dict[str, float], pd.DataFrame]:
    """
    Compute feature weights using outer-train data only.

    Current score:
    - absolute Pearson correlation between feature and DeltaTMS within train data
    - clinical/protected columns are kept at weight 1.0 by default

    This is intentionally simple and conservative. It can be replaced later by
    selection-frequency or coefficient-stability scoring, but this version is safer
    as the first feature-weighting experiment.
    """
    protected_keep_cols = protected_keep_cols or []
    protected_set = set(protected_keep_cols)

    rows = []
    for col in X_train.columns:
        is_protected = col in protected_set

        if FEATURE_WEIGHT_EXCLUDE_CLINICAL and is_protected:
            score = 0.0
            weight = FEATURE_WEIGHT_DEFAULT
        else:
            score = _safe_abs_corr(X_train[col], y_train, min_obs=min_obs)
            weight = np.nan

        rows.append({
            "feature": col,
            "feature_score": float(score),
            "is_protected": bool(is_protected),
            "weight": weight,
        })

    score_df = pd.DataFrame(rows)

    eligible = score_df["weight"].isna()
    if eligible.any():
        s = score_df.loc[eligible, "feature_score"].astype(float)
        s_min = float(s.min())
        s_max = float(s.max())

        if np.isclose(s_min, s_max):
            scaled = pd.Series(0.0, index=s.index)
        else:
            scaled = (s - s_min) / (s_max - s_min)

        score_df.loc[eligible, "weight"] = min_weight + scaled * (max_weight - min_weight)

    score_df["weight"] = score_df["weight"].fillna(FEATURE_WEIGHT_DEFAULT).astype(float)
    score_df = score_df.sort_values(
        ["weight", "feature_score"],
        ascending=[False, False],
    ).reset_index(drop=True)

    weight_map = dict(zip(score_df["feature"].astype(str), score_df["weight"].astype(float)))

    return weight_map, score_df


def build_feature_weights(
    feature_score_df: pd.DataFrame,
    mode: str = "soft",
    top_k: int = 50,
    high_weight: float = 1.5,
    low_weight: float = 0.5,
    min_weight: float = 0.5,
    max_weight: float = 1.5,
):
    """
    Backward-compatible helper.
    Existing code may still call this function.
    """
    df = feature_score_df.copy()

    if len(df) == 0:
        return pd.DataFrame(columns=["feature", "weight"])

    if mode == "topk":
        df["weight"] = low_weight
        top_feats = df.head(top_k)["feature"].tolist()
        df.loc[df["feature"].isin(top_feats), "weight"] = high_weight

    elif mode == "soft":
        s = df["feature_score"].values.astype(float)
        s_min, s_max = s.min(), s.max()
        if np.isclose(s_min, s_max):
            scaled = np.ones_like(s)
        else:
            scaled = (s - s_min) / (s_max - s_min)
        df["weight"] = min_weight + scaled * (max_weight - min_weight)

    else:
        raise ValueError(f"Unknown weighting mode: {mode}")

    return df[["feature", "weight"]].copy()


In [18]:
def extract_final_feature_info(fitted_pipeline: Pipeline) -> pd.DataFrame:
    preprocess = fitted_pipeline.named_steps["preprocess"]
    feature_names = list(preprocess.get_feature_names_out())

    model = fitted_pipeline.named_steps["model"]

    coef = None
    if hasattr(model, "coef_"):
        coef = np.ravel(model.coef_)
    elif hasattr(model, "x_weights_"):
        # PLS does not expose coef_ in exactly the same way across sklearn versions.
        # x_weights_ is not identical to regression coefficients, but it is useful as a trace.
        coef = np.ravel(getattr(model, "x_weights_", np.array([])))

    if coef is not None and len(coef) == len(feature_names):
        coef_list = coef.tolist()
    else:
        coef_list = [np.nan] * len(feature_names)

    feature_df = pd.DataFrame({
        "feature": feature_names,
        "coef": coef_list,
        "abs_coef": np.abs(coef_list),
    })

    if USE_FEATURE_WEIGHTING and "feature_weight" in fitted_pipeline.named_steps:
        fw = fitted_pipeline.named_steps["feature_weight"]
        weight_by_processed = dict(zip(getattr(fw, "feature_names_", []), getattr(fw, "weights_", [])))
        feature_df["feature_weight"] = feature_df["feature"].map(weight_by_processed).fillna(FEATURE_WEIGHT_DEFAULT)
    else:
        feature_df["feature_weight"] = FEATURE_WEIGHT_DEFAULT

    feature_df = feature_df.sort_values(
        "abs_coef",
        ascending=False,
        na_position="last",
    ).reset_index(drop=True)

    return feature_df


def make_regression_strata(
    y: pd.Series,
    n_bins: int,
    n_splits: int,
) -> pd.Series:
    """
    Quantile-based strata for regression CV.
    """
    y_num = pd.to_numeric(y, errors="coerce")

    if y_num.notna().sum() < n_splits:
        return pd.Series(index=y.index, data=0).astype(int)

    max_bins = min(n_bins, int(y_num.nunique()))

    for bins in range(max_bins, 1, -1):
        try:
            y_bin = pd.qcut(
                y_num,
                q=bins,
                labels=False,
                duplicates="drop",
            )
            y_bin = pd.Series(y_bin, index=y.index)

            counts = y_bin.value_counts(dropna=True)

            if len(counts) >= 2 and counts.min() >= n_splits:
                return y_bin.fillna(-1).astype(int)

        except Exception:
            continue

    return pd.Series(index=y.index, data=0).astype(int)


def make_outer_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Outer CV split helper.
    """
    if USE_STRATIFIED_OUTER_CV and X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=n_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            outer_cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=OUTER_CV_RANDOM_STATE,
            )
            return list(outer_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    outer_cv = GroupKFold(n_splits=n_splits)
    return list(outer_cv.split(X, y, groups=groups)), "GroupKFold_fallback"


def make_inner_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    groups: np.ndarray,
    n_splits: int,
):
    """
    Inner CV split helper.

    목적:
    - model/alpha/top-k 선택이 특정 fold의 y 분포 때문에 흔들리는 것을 줄인다.
    - patient index가 unique이면 y-bin stratified split을 사용한다.
    - 실패하면 GroupKFold로 fallback한다.
    """
    if X.index.is_unique:
        y_bins = make_regression_strata(
            y=y,
            n_bins=N_Y_BINS_FOR_OUTER_CV,
            n_splits=n_splits,
        )

        if y_bins.nunique(dropna=True) >= 2:
            inner_cv = StratifiedKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=RANDOM_STATE,
            )
            return list(inner_cv.split(X, y_bins)), "StratifiedKFold_y_bins"

    inner_cv = GroupKFold(n_splits=n_splits)
    return list(inner_cv.split(X, y, groups=groups)), "GroupKFold_fallback"


def fit_best_model_only(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    candidate_models: list[str],
    n_splits_inner: int = 3,
    protected_keep_cols: list[str] | None = None,
    feature_selection_mode: str = "f_regression_topk",
    top_k_after_sparse: int | None = 100,
):
    """
    Select model/hyperparameters using train-side inner OOF R².

    Paper-style leakage rule:
    - In each inner fold, feature selection is fitted only on inner-train.
    - Inner-validation y is never used for feature selection or feature weighting.
    - Final feature selection is fitted only on the full predefined training set.
    - Holdout test y is never used until final evaluation.
    """
    protected_keep_cols = protected_keep_cols or []

    search_rows = []
    best_obj = None
    best_score = -np.inf

    inner_splits, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )

    for model_type in candidate_models:
        grid = adjust_param_grid_for_training_data(
            X_train=X_train,
            y_train=y_train,
            model_type=model_type,
        )

        for params in grid:
            inner_oof = pd.Series(index=X_train.index, dtype=float)
            inner_fold_scores = []
            inner_n_features = []

            for inner_fold, (tr_idx, va_idx) in enumerate(inner_splits, start=1):
                X_tr_raw = X_train.iloc[tr_idx].copy()
                y_tr = y_train.iloc[tr_idx].copy()
                X_va_raw = X_train.iloc[va_idx].copy()
                y_va = y_train.iloc[va_idx].copy()

                X_tr, X_va, _, selected_features = select_features_train_valid(
                    X_train=X_tr_raw,
                    X_valid=X_va_raw,
                    y_train=y_tr,
                    k=top_k_after_sparse,
                    mode=feature_selection_mode,
                    protected_keep_cols=protected_keep_cols,
                )
                inner_n_features.append(len(selected_features))

                if USE_FEATURE_WEIGHTING:
                    inner_weight_map, _ = compute_feature_weight_map_from_train(
                        X_train=X_tr,
                        y_train=y_tr,
                        protected_keep_cols=[c for c in protected_keep_cols if c in X_tr.columns],
                    )
                else:
                    inner_weight_map = None

                pipe = make_pipeline(
                    X_tr,
                    model_type=model_type,
                    feature_weight_map=inner_weight_map,
                )
                pipe.set_params(**params)
                pipe.fit(X_tr, y_tr)

                pred = predict_1d(pipe, X_va)
                inner_oof.iloc[va_idx] = pred

                try:
                    inner_fold_scores.append(float(r2_score(y_va, pred)))
                except Exception:
                    inner_fold_scores.append(np.nan)

            valid_mask = inner_oof.notna()

            if valid_mask.sum() >= 2:
                inner_oof_r2 = float(
                    r2_score(
                        y_train.loc[valid_mask],
                        inner_oof.loc[valid_mask],
                    )
                )
                inner_oof_mae = float(
                    mean_absolute_error(
                        y_train.loc[valid_mask],
                        inner_oof.loc[valid_mask],
                    )
                )
            else:
                inner_oof_r2 = np.nan
                inner_oof_mae = np.nan

            inner_mean_fold_r2 = (
                float(np.nanmean(inner_fold_scores))
                if len(inner_fold_scores) > 0 and not np.all(pd.isna(inner_fold_scores))
                else np.nan
            )

            inner_std_fold_r2 = (
                float(np.nanstd(inner_fold_scores))
                if len(inner_fold_scores) > 0 and not np.all(pd.isna(inner_fold_scores))
                else np.nan
            )

            if pd.notna(inner_oof_r2):
                selection_score = float(inner_oof_r2)
                if pd.notna(inner_std_fold_r2):
                    selection_score -= INNER_SELECTION_STD_PENALTY * float(inner_std_fold_r2)
            else:
                selection_score = np.nan

            search_rows.append({
                "model_type": model_type,
                "params": json.dumps(params, default=str),
                "inner_mean_r2": inner_oof_r2,
                "inner_oof_r2": inner_oof_r2,
                "inner_oof_mae": inner_oof_mae,
                "inner_mean_fold_r2": inner_mean_fold_r2,
                "inner_std_fold_r2": inner_std_fold_r2,
                "selection_score_stability_penalized": selection_score,
                "inner_cv_type": inner_cv_type,
                "feature_selection_mode": feature_selection_mode,
                "top_k_after_sparse": top_k_after_sparse,
                "mean_inner_selected_features": (
                    float(np.mean(inner_n_features)) if len(inner_n_features) > 0 else np.nan
                ),
                "min_inner_selected_features": (
                    int(np.min(inner_n_features)) if len(inner_n_features) > 0 else np.nan
                ),
                "max_inner_selected_features": (
                    int(np.max(inner_n_features)) if len(inner_n_features) > 0 else np.nan
                ),
                "use_feature_weighting": USE_FEATURE_WEIGHTING,
                "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
                "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
            })

            if pd.notna(selection_score) and selection_score > best_score:
                best_score = selection_score
                best_obj = {
                    "model_type": model_type,
                    "params": params,
                }

    if best_obj is None:
        raise ValueError("No valid model/parameter combination found in fit_best_model_only")

    # Final train-only feature selection for the holdout test model.
    X_train_sel, _, final_selection_df, selected_features = select_features_train_valid(
        X_train=X_train,
        X_valid=X_train,
        y_train=y_train,
        k=top_k_after_sparse,
        mode=feature_selection_mode,
        protected_keep_cols=protected_keep_cols,
    )

    if USE_FEATURE_WEIGHTING:
        final_weight_map, final_weight_df = compute_feature_weight_map_from_train(
            X_train=X_train_sel,
            y_train=y_train,
            protected_keep_cols=[c for c in protected_keep_cols if c in X_train_sel.columns],
        )
    else:
        final_weight_map = None
        final_weight_df = pd.DataFrame(columns=["feature", "feature_score", "is_protected", "weight"])

    final_pipe = make_pipeline(
        X_train_sel,
        model_type=best_obj["model_type"],
        feature_weight_map=final_weight_map,
    )
    final_pipe.set_params(**best_obj["params"])
    final_pipe.fit(X_train_sel, y_train)

    search_df = pd.DataFrame(search_rows)
    if len(search_df) > 0:
        search_df = search_df.sort_values(
            "selection_score_stability_penalized",
            ascending=False,
            na_position="last",
        ).reset_index(drop=True)

    return {
        "best_model_type": best_obj["model_type"],
        "best_params": best_obj["params"],
        "best_score": best_score,
        "search_df": search_df,
        "fitted_pipeline": final_pipe,
        "feature_weight_df": final_weight_df,
        "feature_selection_df": final_selection_df,
        "selected_features": selected_features,
    }


def get_inner_oof_predictions_model_only(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    groups_train: np.ndarray,
    candidate_models: list[str],
    n_splits_inner: int = 3,
    protected_keep_cols: list[str] | None = None,
    feature_selection_mode: str = "f_regression_topk",
    top_k_after_sparse: int | None = 100,
):
    """
    Diagnostic train-inner OOF predictions using the same paper-style fold-safe selection.
    """
    inner_splits, inner_cv_type = make_inner_cv_splits(
        X=X_train,
        y=y_train,
        groups=groups_train,
        n_splits=n_splits_inner,
    )

    oof = pd.Series(index=X_train.index, dtype=float)

    for tr_idx, va_idx in inner_splits:
        X_tr = X_train.iloc[tr_idx].copy()
        y_tr = y_train.iloc[tr_idx].copy()
        X_va = X_train.iloc[va_idx].copy()

        fit_result = fit_best_model_only(
            X_train=X_tr,
            y_train=y_tr,
            groups_train=groups_train[tr_idx],
            candidate_models=candidate_models,
            n_splits_inner=n_splits_inner,
            protected_keep_cols=protected_keep_cols,
            feature_selection_mode=feature_selection_mode,
            top_k_after_sparse=top_k_after_sparse,
        )

        selected_features = fit_result["selected_features"]
        X_va_sel = X_va.reindex(columns=selected_features).copy()

        best_pipe = fit_result["fitted_pipeline"]
        pred_va = predict_1d(best_pipe, X_va_sel)

        oof.iloc[va_idx] = pred_va

    return oof


In [19]:
def generate_weight_grid(n_models: int, step: float = 0.1):
    """
    Generate nonnegative weight tuples that sum to 1.
    Example:
      n_models=2, step=0.5 -> [(0.0,1.0),(0.5,0.5),(1.0,0.0)]
    """
    if n_models < 1:
        raise ValueError("n_models must be >= 1")
    if step <= 0 or step > 1:
        raise ValueError("step must be in (0, 1]")

    step_int = int(round(1 / step))
    if not np.isclose(step_int * step, 1.0):
        raise ValueError("step must divide 1.0 exactly, e.g. 0.5, 0.25, 0.2, 0.1")

    grids = []

    def backtrack(prefix, remaining, depth):
        if depth == n_models - 1:
            grids.append(tuple(prefix + [remaining / step_int]))
            return
        for v in range(remaining + 1):
            backtrack(prefix + [v / step_int], remaining - v, depth + 1)

    backtrack([], step_int, 0)
    return grids

In [20]:
def encode_clinical_train_valid(
    X_train: pd.DataFrame,
    X_valid: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    """
    Clinical을 항상 fold 내부에서 encoding한다.
    반환:
      - encoded X_train
      - encoded X_valid
      - encoded clinical column names
    """
    base_clin_cols = [c for c in get_clinical_columns() if c in X_train.columns]
    if len(base_clin_cols) == 0:
        return X_train, X_valid, []

    tr_clin = X_train.loc[:, base_clin_cols].copy()
    va_clin = X_valid.loc[:, [c for c in base_clin_cols if c in X_valid.columns]].copy()

    tr_other = X_train.drop(columns=base_clin_cols, errors="ignore").copy()
    va_other = X_valid.drop(columns=base_clin_cols, errors="ignore").copy()

    if "Age" in tr_clin.columns:
        tr_clin["Age"] = pd.to_numeric(tr_clin["Age"], errors="coerce")
    if "Age" in va_clin.columns:
        va_clin["Age"] = pd.to_numeric(va_clin["Age"], errors="coerce")

    cat_cols = [c for c in ["Gender", "Level"] if c in tr_clin.columns]

    if len(cat_cols) > 0:
        tr_clin_enc = pd.get_dummies(
            tr_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = pd.get_dummies(
            va_clin,
            columns=cat_cols,
            dummy_na=True,
            dtype=float,
        )
        va_clin_enc = va_clin_enc.reindex(columns=tr_clin_enc.columns, fill_value=0.0)
    else:
        tr_clin_enc = tr_clin.copy()
        va_clin_enc = va_clin.copy()

    tr_clin_enc = tr_clin_enc.apply(pd.to_numeric, errors="coerce")
    va_clin_enc = va_clin_enc.apply(pd.to_numeric, errors="coerce")

    encoded_clin_cols = tr_clin_enc.columns.tolist()

    X_train_out = pd.concat([tr_other, tr_clin_enc], axis=1)
    X_valid_out = pd.concat([va_other, va_clin_enc], axis=1)

    return X_train_out, X_valid_out, encoded_clin_cols

In [21]:
def run_single_omics_final_train_test(
    X: pd.DataFrame,
    y: pd.Series,
    tissue: str,
    combo: str,
    tp: int,
    train_ids: list[str],
    test_ids: list[str],
    candidate_models: list[str],
    feature_mode: str,
    clinical_mode: str = "light",
    feature_selection_mode: str = "f_regression_topk",
    min_obs_frac: float = MIN_OBS_FRAC,
    top_k_after_sparse: int | None = 100,
):
    train_ids_in = [pid for pid in train_ids if pid in X.index]
    test_ids_in = [pid for pid in test_ids if pid in X.index]

    X_train = X.loc[train_ids_in].copy()
    y_train = y.loc[train_ids_in].copy()
    X_test = X.loc[test_ids_in].copy()
    y_test = y.loc[test_ids_in].copy()

    print("\n[TRAIN / TEST SPLIT CHECK]")
    print("clinical_mode:", clinical_mode)
    print("feature_selection_mode:", feature_selection_mode)
    print("top_k_after_sparse:", top_k_after_sparse)
    print("X_train shape:", X_train.shape)
    print("X_test  shape:", X_test.shape)
    print("y_train shape:", y_train.shape)
    print("y_test  shape:", y_test.shape)

    train_ids_set = set(X_train.index.astype(str))
    test_ids_set = set(X_test.index.astype(str))
    overlap_ids = sorted(train_ids_set.intersection(test_ids_set))

    print("train unique ids:", len(train_ids_set))
    print("test unique ids :", len(test_ids_set))
    print("overlap ids count:", len(overlap_ids))
    print("overlap ids example:", overlap_ids[:10])

    delta_cols = [c for c in X_train.columns if "_d" in str(c)]
    print("  delta cols count:", len(delta_cols))
    print("  delta examples:", delta_cols[:10])

    clin_cols_present = [c for c in get_clinical_columns() if c in X_train.columns]
    print("  clinical cols present:", clin_cols_present)

    if len(X_train) < N_SPLITS_INNER:
        raise ValueError("not enough training samples for inner CV")
    if len(X_test) == 0:
        raise ValueError("no holdout test samples available")

    protected_keep_cols = get_clinical_columns() if clinical_mode == "light" else []

    X_train_f, X_test_f = filter_sparse_features_train_test(
        X_train=X_train,
        X_valid=X_test,
        min_obs_frac=min_obs_frac,
        protected_keep_cols=protected_keep_cols,
    )

    print("\n[AFTER SPARSE FILTER]")
    print("X_train_f shape:", X_train_f.shape)
    print("X_test_f  shape:", X_test_f.shape)

    removed_cols = [c for c in X_train.columns if c not in X_train_f.columns]
    print("n_removed_by_sparse:", len(removed_cols))
    print("removed example:", removed_cols[:20])

    remaining_clin = [c for c in get_clinical_columns() if c in X_train_f.columns]
    print("clinical remaining after sparse:", remaining_clin)

    n_before_sparse = X_train.shape[1]
    n_after_sparse = X_train_f.shape[1]

    if X_train_f.shape[1] == 0:
        raise ValueError("no features left after sparse filtering")

    if clinical_mode == "light":
        X_train_f, X_test_f, encoded_clin_cols = encode_clinical_train_valid(
            X_train=X_train_f,
            X_valid=X_test_f,
        )
        protected_keep_cols = encoded_clin_cols
    else:
        encoded_clin_cols = []
        protected_keep_cols = []

    print("\n[AFTER CLINICAL ENCODING]")
    print("X_train_f shape:", X_train_f.shape)
    print("X_test_f  shape:", X_test_f.shape)
    print("encoded clinical cols:", encoded_clin_cols[:20])
    print("n_encoded_clinical_cols:", len(encoded_clin_cols))

    non_numeric_cols = X_train_f.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
    print("non_numeric_cols_after_encoding:", non_numeric_cols[:20])
    print("n_non_numeric_after_encoding:", len(non_numeric_cols))    

    if len(non_numeric_cols) > 0:
        raise TypeError(f"Non-numeric columns remain after encoding: {non_numeric_cols[:20]}")

    # IMPORTANT:
    # Do NOT apply target-aware top-k here.
    # Feature selection is now performed inside fit_best_model_only for each inner fold.
    # This prevents inner-CV leakage and follows the paper-style CV rule.
    n_after_pre_selection = X_train_f.shape[1]

    print("\n[BEFORE FOLD-SAFE SELECTION]")
    print("X_train_f shape:", X_train_f.shape)
    print("X_test_f  shape:", X_test_f.shape)
    print("candidate feature examples:", X_train_f.columns[:30].tolist())

    pd.DataFrame({
        "feature": X_train_f.columns,
        "has_delta": ["_d" in str(c) for c in X_train_f.columns],
        "is_clinical": [c in protected_keep_cols for c in X_train_f.columns],
    }).to_csv(
        FEATURE_TRACE_DIR / f"{feature_mode}_{clinical_mode}_{feature_selection_mode}_{tissue}_{combo}_{tp}_before_fold_safe_selection.csv",
        index=False,
    )

    fit_result = fit_best_model_only(
        X_train=X_train_f,
        y_train=y_train,
        groups_train=X_train_f.index.to_numpy(),
        candidate_models=candidate_models,
        n_splits_inner=N_SPLITS_INNER,
        protected_keep_cols=protected_keep_cols,
        feature_selection_mode=feature_selection_mode,
        top_k_after_sparse=top_k_after_sparse,
    )

    selected_features = fit_result["selected_features"]
    X_train_model = X_train_f.loc[:, selected_features].copy()
    X_test_model = X_test_f.reindex(columns=selected_features).copy()

    print("\n[AFTER FINAL TRAIN-ONLY FEATURE SELECTION]")
    print("selected n_features:", len(selected_features))
    print("selected feature examples:", selected_features[:30])

    best_pipe = fit_result["fitted_pipeline"]

    pred_train = predict_1d(best_pipe, X_train_model)
    pred_test = predict_1d(best_pipe, X_test_model)

    train_r2 = r2_score(y_train, pred_train)
    train_mae = mean_absolute_error(y_train, pred_train)

    test_r2 = r2_score(y_test, pred_test)
    test_mae = mean_absolute_error(y_test, pred_test)

    pred_df = pd.DataFrame({
        "Patient": X_test_model.index,
        "y_true": y_test.values,
        "y_pred": pred_test,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "best_model_type": fit_result["best_model_type"],
        "top_k_after_sparse": top_k_after_sparse,
    })

    train_pred_df = pd.DataFrame({
        "Patient": X_train_model.index,
        "y_true": y_train.values,
        "y_pred": pred_train,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "best_model_type": fit_result["best_model_type"],
        "top_k_after_sparse": top_k_after_sparse,
    })

    print("\n[FINAL PREDICTION CHECK]")
    print("best_model_type:", fit_result["best_model_type"])
    print("best_params:", fit_result["best_params"])
    print("best_score_inner_oof_r2:", fit_result["best_score"])

    print("train_r2:", train_r2)
    print("train_mae:", train_mae)
    print("test_r2 :", test_r2)
    print("test_mae:", test_mae)

    print("pred_train summary:")
    print("  mean:", float(np.mean(pred_train)))
    print("  std :", float(np.std(pred_train)))
    print("  min :", float(np.min(pred_train)))
    print("  max :", float(np.max(pred_train)))

    print("pred_test summary:")
    print("  mean:", float(np.mean(pred_test)))
    print("  std :", float(np.std(pred_test)))
    print("  min :", float(np.min(pred_test)))
    print("  max :", float(np.max(pred_test)))

    print("y_test summary:")
    print("  mean:", float(np.mean(y_test)))
    print("  std :", float(np.std(y_test)))
    print("  min :", float(np.min(y_test)))
    print("  max :", float(np.max(y_test)))

    feature_info_df = extract_final_feature_info(best_pipe).copy()
    feature_weight_df = fit_result.get("feature_weight_df", pd.DataFrame()).copy()
    feature_selection_df = fit_result.get("feature_selection_df", pd.DataFrame()).copy()

    for trace_df in [feature_weight_df, feature_selection_df]:
        if isinstance(trace_df, pd.DataFrame) and len(trace_df) > 0:
            trace_df["feature_mode"] = feature_mode
            trace_df["clinical_mode"] = clinical_mode
            trace_df["feature_selection_mode"] = feature_selection_mode
            trace_df["tissue"] = tissue
            trace_df["combo"] = combo
            trace_df["tp"] = tp
            trace_df["experiment_tag"] = EXPERIMENT_TAG

    metrics = {
        "experiment_tag": EXPERIMENT_TAG,
        "use_feature_weighting": USE_FEATURE_WEIGHTING,
        "feature_weight_score_mode": FEATURE_WEIGHT_SCORE_MODE if USE_FEATURE_WEIGHTING else "none",
        "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
        "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
        "mean_feature_weight": (
            float(feature_weight_df["weight"].mean())
            if len(feature_weight_df) > 0 and "weight" in feature_weight_df.columns
            else FEATURE_WEIGHT_DEFAULT
        ),
        "max_feature_weight": (
            float(feature_weight_df["weight"].max())
            if len(feature_weight_df) > 0 and "weight" in feature_weight_df.columns
            else FEATURE_WEIGHT_DEFAULT
        ),
        "n_weighted_features": (
            int((feature_weight_df["weight"] > FEATURE_WEIGHT_DEFAULT).sum())
            if len(feature_weight_df) > 0 and "weight" in feature_weight_df.columns
            else 0
        ),
        "feature_mode": feature_mode,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "best_model_type": fit_result["best_model_type"],
        "best_score_inner_cv_r2": float(fit_result.get("best_score", np.nan)),
        "selection_basis": "training_matched_inner_oof_r2_with_fold_safe_feature_selection_not_test_r2",
        "n_train": len(X_train_model),
        "n_test": len(X_test_model),
        "n_features_before_sparse": n_before_sparse,
        "n_features_after_sparse": n_after_sparse,
        "n_features_before_fold_safe_selection": n_after_pre_selection,
        "n_features_after_selection": len(selected_features),
        "n_features_after_topk": len(selected_features),
        "top_k_after_sparse": top_k_after_sparse,
        "train_r2": train_r2,
        "train_mae": train_mae,
        "test_r2": test_r2,
        "test_mae": test_mae,
    }

    return {
        "metrics": metrics,
        "search_df": fit_result["search_df"],
        "train_pred_df": train_pred_df,
        "test_pred_df": pred_df,
        "feature_df": feature_info_df,
        "feature_weight_df": feature_weight_df,
        "feature_selection_df": feature_selection_df,
        "best_model_type": fit_result["best_model_type"],
        "best_params": fit_result["best_params"],
        "selected_features": selected_features,
    }


In [22]:
def run_one_single_omics_experiment(
    tissue: str,
    combo: str,
    tp: int,
    y_all: pd.Series,
    feature_mode: str,
    candidate_models: list[str],
    clinical_mode: str = "light",
    feature_selection_mode: str = "f_regression_topk",
    top_k_after_sparse_override: int | None = 100,
):
    feature_path = build_feature_path(
        feature_mode=feature_mode,
        tissue=tissue,
        combo=combo,
        tp=tp,
    )

    if not feature_path.exists():
        print("missing:", feature_path)
        return None

    print(
        f"\n[RUN-FINAL-TEST] feature_mode={feature_mode}, "
        f"clinical_mode={clinical_mode}, selection={feature_selection_mode}, "
        f"tissue={tissue}, combo={combo}, tp={tp}"
    )
    print("feature_path:", feature_path)

    X = load_feature_csv(feature_path)
    X, y = align_xy(X, y_all)

    print("\n[EXPERIMENT START]")
    print("feature_mode:", feature_mode)
    print("clinical_mode:", clinical_mode)
    print("feature_selection_mode:", feature_selection_mode)
    print("tissue:", tissue, "| combo:", combo, "| tp:", tp)
    print("X shape before clinical:", X.shape)
    print("y shape:", y.shape)

    delta_cols = [c for c in X.columns if "_d" in str(c)]
    print("n_delta_features:", len(delta_cols))
    print("delta examples:", delta_cols[:10])

    print("y summary:")
    print("  mean:", float(y.mean()))
    print("  std :", float(y.std()))
    print("  min :", float(y.min()))
    print("  max :", float(y.max()))

    if clinical_mode == "light":
        X = merge_with_light_clinical(
            X_feat=X,
            tissue=tissue,
            combo=combo,
            tp=tp,
        )
        assert_no_forbidden_proxy_columns(X, use_light_clinical=True)
    elif clinical_mode == "omics_only":
        assert_no_forbidden_proxy_columns(X, use_light_clinical=False)
    else:
        raise ValueError(f"Unknown clinical_mode: {clinical_mode}")

    delta_cols = [c for c in X.columns if "_d" in str(c)]
    clinical_cols_present = [c for c in get_clinical_columns() if c in X.columns]

    print("X shape after clinical mode:", X.shape)
    print("delta cols count:", len(delta_cols))
    print("clinical cols present:", clinical_cols_present)

    train_ids_in = [pid for pid in TRAIN_IDS if pid in X.index]
    test_ids_in = [pid for pid in TEST_IDS if pid in X.index]

    base_skip = {
        "status": "skipped",
        "experiment_tag": EXPERIMENT_TAG,
        "use_feature_weighting": USE_FEATURE_WEIGHTING,
        "feature_weight_score_mode": FEATURE_WEIGHT_SCORE_MODE if USE_FEATURE_WEIGHTING else "none",
        "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
        "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
        "mean_feature_weight": np.nan,
        "max_feature_weight": np.nan,
        "n_weighted_features": np.nan,
        "feature_mode": feature_mode,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "best_model_type": None,
        "n_train_available": len(train_ids_in),
        "n_test_available": len(test_ids_in),
        "test_r2": np.nan,
        "test_mae": np.nan,
        "train_r2": np.nan,
        "train_mae": np.nan,
        "n_features_input": X.shape[1],
        "n_delta_features_input": len(delta_cols),
        "n_clinical_input": len(clinical_cols_present),
        "top_k_after_sparse": top_k_after_sparse_override,
        "selection_basis": "not_run",
    }

    if len(train_ids_in) < N_SPLITS_INNER:
        print("skip: not enough training patients after TRAIN_IDS filter")
        out = base_skip.copy()
        out["reason"] = "not enough training patients after TRAIN_IDS filter"
        return out

    if len(test_ids_in) == 0:
        print("skip: no test patients after TEST_IDS filter")
        out = base_skip.copy()
        out["reason"] = "no test patients after TEST_IDS filter"
        return out

    top_k_after_sparse = top_k_after_sparse_override if top_k_after_sparse_override is not None else 100

    result = run_single_omics_final_train_test(
        X=X,
        y=y,
        tissue=tissue,
        combo=combo,
        tp=tp,
        train_ids=TRAIN_IDS,
        test_ids=TEST_IDS,
        candidate_models=candidate_models,
        feature_mode=feature_mode,
        clinical_mode=clinical_mode,
        feature_selection_mode=feature_selection_mode,
        min_obs_frac=MIN_OBS_FRAC,
        top_k_after_sparse=top_k_after_sparse,
    )

    prefix = f"{feature_mode}_{clinical_mode}_{feature_selection_mode}_{tissue}_{combo}_{tp}_topk{top_k_after_sparse}"

    pd.DataFrame([result["metrics"]]).to_csv(
        SAVE_DIR / f"{prefix}_final_test_summary.csv",
        index=False
    )
    result["search_df"].to_csv(
        SAVE_DIR / f"{prefix}_final_test_search_results.csv",
        index=False
    )
    result["train_pred_df"].to_csv(
        SAVE_DIR / f"{prefix}_final_train_predictions.csv",
        index=False
    )
    result["test_pred_df"].to_csv(
        SAVE_DIR / f"{prefix}_final_test_predictions.csv",
        index=False
    )
    result["feature_df"].to_csv(
        SAVE_DIR / f"{prefix}_final_selected_features.csv",
        index=False
    )
    if isinstance(result.get("feature_weight_df", None), pd.DataFrame) and len(result["feature_weight_df"]) > 0:
        result["feature_weight_df"].to_csv(
            SAVE_DIR / f"{prefix}_final_feature_weights.csv",
            index=False
        )
    if isinstance(result.get("feature_selection_df", None), pd.DataFrame) and len(result["feature_selection_df"]) > 0:
        result["feature_selection_df"].to_csv(
            SAVE_DIR / f"{prefix}_final_feature_selection_scores.csv",
            index=False
        )

    return {
        "status": "ok",
        "experiment_tag": EXPERIMENT_TAG,
        "use_feature_weighting": USE_FEATURE_WEIGHTING,
        "feature_weight_score_mode": FEATURE_WEIGHT_SCORE_MODE if USE_FEATURE_WEIGHTING else "none",
        "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
        "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
        "mean_feature_weight": result["metrics"].get("mean_feature_weight", np.nan),
        "max_feature_weight": result["metrics"].get("max_feature_weight", np.nan),
        "n_weighted_features": result["metrics"].get("n_weighted_features", np.nan),
        "feature_mode": feature_mode,
        "clinical_mode": clinical_mode,
        "feature_selection_mode": feature_selection_mode,
        "tissue": tissue,
        "combo": combo,
        "tp": tp,
        "reason": None,
        "best_model_type": result["best_model_type"],
        "n_train": result["metrics"]["n_train"],
        "n_test": result["metrics"]["n_test"],
        "train_r2": result["metrics"]["train_r2"],
        "train_mae": result["metrics"]["train_mae"],
        "test_r2": result["metrics"]["test_r2"],
        "test_mae": result["metrics"]["test_mae"],
        "best_score_inner_cv_r2": result["metrics"]["best_score_inner_cv_r2"],
        "selection_basis": result["metrics"].get("selection_basis", "training_matched_inner_oof_r2_with_fold_safe_feature_selection_not_test_r2"),
        "n_features_input": X.shape[1],
        "n_delta_features_input": len(delta_cols),
        "n_clinical_input": len(clinical_cols_present),
        "n_features_after_selection": result["metrics"].get("n_features_after_selection", np.nan),
        "top_k_after_sparse": top_k_after_sparse,
    }


In [23]:
# ==========================================
# Sanity check before running Experiment B
# ==========================================
y_all = load_target(TARGET_PATH)

for tissue, combo, tp in product(TISSUES, COMBOS, TIMEPOINTS):
    feature_path = build_feature_path(feature_mode="omics", tissue=tissue, combo=combo, tp=tp)
    if not feature_path.exists():
        print("missing:", feature_path.name)
        continue

    X = load_feature_csv(feature_path)
    X, y = align_xy(X, y_all)
    
    

    delta_cols = [c for c in X.columns if "_d" in str(c)]
    print(
        f"{feature_path.name}: shape={X.shape}, delta_cols={len(delta_cols)}"
    )

    if len(delta_cols) > 0:
        print("  delta examples:", delta_cols[:10])

x_omics_csf_A_24.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_48.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_72.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_96.csv: shape=(103, 438), delta_cols=0
x_omics_csf_A_120.csv: shape=(103, 438), delta_cols=0
x_omics_csf_B_24.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_48.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_72.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_96.csv: shape=(83, 749), delta_cols=0
x_omics_csf_B_120.csv: shape=(83, 749), delta_cols=0
x_omics_csf_C_24.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_48.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_72.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_96.csv: shape=(39, 312), delta_cols=0
x_omics_csf_C_120.csv: shape=(39, 312), delta_cols=0
x_omics_ser_A_24.csv: shape=(108, 438), delta_cols=0
x_omics_ser_A_48.csv: shape=(108, 438), delta_cols=0
x_omics_ser_A_72.csv: shape=(108, 438), delta_cols=0
x_omics_ser_A_96.csv: shape=(108, 438), delta_cols=0


In [24]:
y_all = load_target(TARGET_PATH)

FEATURE_MODE_TO_RUN = "omics"

# ==========================================
# Select held-out test configs from TRAINING summary only
# ==========================================
def _first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def load_and_select_single_test_configs(
    train_summary_path: Path,
    selection_mode: str = "top_n_per_tissue_tp",
    top_n_per_tissue_tp: int = 5,
    top_k_per_tissue_combo_tp: int = 3,
) -> pd.DataFrame:
    """Select holdout-test configs using training OOF/CV only.

    Important:
    - This function never uses holdout-test metrics for selection.
    - top_n_per_tissue_tp is a sensitivity-analysis mode. It evaluates several
      training-selected candidates per tissue+TP to see whether OOF-best is unstable.
    - Any test-best table saved later is descriptive only and must not be used as
      the final model-selection criterion.
    """
    if not train_summary_path.exists():
        raise FileNotFoundError(
            f"Training summary not found: {train_summary_path}\n"
            "Run the single-omics training notebook first, or update TRAIN_SUMMARY_PATH."
        )

    train_summary = pd.read_csv(train_summary_path)
    print("loaded training summary:", train_summary_path)
    print("training summary shape:", train_summary.shape)

    if "status" in train_summary.columns:
        train_summary = train_summary[train_summary["status"].astype(str).str.lower().eq("ok")].copy()
        print("training ok rows:", len(train_summary))

    required = ["tissue", "combo", "tp", "clinical_mode", "feature_selection_mode", "top_k_after_sparse"]
    missing = [c for c in required if c not in train_summary.columns]
    if missing:
        raise ValueError(f"Training summary is missing required columns: {missing}")

    model_col = _first_existing_col(
        train_summary,
        ["final_full_train_best_model_type", "best_model_type", "model_type"],
    )
    if model_col is None:
        raise ValueError(
            "Training summary must contain one of: "
            "final_full_train_best_model_type, best_model_type, model_type"
        )

    train_summary = train_summary.copy()
    train_summary["selected_model_type_from_training"] = train_summary[model_col].astype(str)

    numeric_cols = [
        "oof_r2", "mean_valid_r2", "std_valid_r2", "mean_train_r2", "oof_mae",
        "final_full_train_best_score_inner_oof_r2", "final_full_train_r2", "final_full_train_mae",
        "top_k_after_sparse",
    ]
    for c in numeric_cols:
        if c in train_summary.columns:
            train_summary[c] = pd.to_numeric(train_summary[c], errors="coerce")

    if "single_selection_score" not in train_summary.columns:
        train_summary["single_selection_score"] = train_summary.get("oof_r2", np.nan)

    sort_by = []
    ascending = []
    for c in TRAIN_SELECTION_SORT_COLUMNS:
        if c in train_summary.columns:
            sort_by.append(c)
            ascending.append(True if c.endswith("mae") else False)

    if len(sort_by) == 0:
        raise ValueError("No usable training-CV ranking columns found in training summary.")

    ranked = train_summary.sort_values(
        by=sort_by,
        ascending=ascending,
        na_position="last",
    ).reset_index(drop=True)

    # Add training-only ranks. These ranks are computed BEFORE any holdout-test evaluation.
    ranked["training_rank_global"] = np.arange(1, len(ranked) + 1)
    ranked["training_rank_within_tissue_tp"] = (
        ranked.groupby(["tissue", "tp"]).cumcount() + 1
    )
    ranked["training_rank_within_tissue_combo_tp"] = (
        ranked.groupby(["tissue", "combo", "tp"]).cumcount() + 1
    )

    if selection_mode == "best_per_tissue_tp":
        selected = ranked[ranked["training_rank_within_tissue_tp"].eq(1)].copy()
    elif selection_mode == "top_n_per_tissue_tp":
        selected = ranked[ranked["training_rank_within_tissue_tp"].le(int(top_n_per_tissue_tp))].copy()
    elif selection_mode == "best_per_tissue_combo_tp":
        selected = ranked[ranked["training_rank_within_tissue_combo_tp"].eq(1)].copy()
    elif selection_mode == "top_k_per_tissue_combo_tp":
        selected = ranked[ranked["training_rank_within_tissue_combo_tp"].le(int(top_k_per_tissue_combo_tp))].copy()
    elif selection_mode == "all_train_ok":
        selected = ranked.copy()
    else:
        raise ValueError(f"Unknown TEST_CONFIG_SELECTION_MODE: {selection_mode}")

    selected = selected.reset_index(drop=True)
    selected["test_selection_basis"] = (
        "selected_before_holdout_test_using_training_only_oof_cv_"
        + selection_mode
    )
    selected["training_model_col_used"] = model_col
    selected["train_experiment_tag"] = TRAIN_EXPERIMENT_TAG
    selected["test_experiment_tag"] = TEST_EXPERIMENT_TAG
    selected["holdout_test_usage_note"] = (
        "Holdout test is evaluation only. Do not choose the final model by test_r2. "
        "For final reporting, use training_rank_within_tissue_tp == 1 unless explicitly doing sensitivity analysis."
    )

    return selected


selected_configs = load_and_select_single_test_configs(
    TRAIN_SUMMARY_PATH,
    selection_mode=TEST_CONFIG_SELECTION_MODE,
    top_n_per_tissue_tp=TOP_N_PER_TISSUE_TP,
    top_k_per_tissue_combo_tp=TOP_K_PER_TISSUE_COMBO_TP,
)

selected_configs_path = SAVE_DIR / f"single_omics_selected_configs_used_for_test_{TEST_EXPERIMENT_TAG}.csv"
selected_configs.to_csv(selected_configs_path, index=False)
print("selected configs saved:", selected_configs_path)
print("n_selected_configs:", len(selected_configs))
print("selection mode:", TEST_CONFIG_SELECTION_MODE)
if TEST_CONFIG_SELECTION_MODE == "top_n_per_tissue_tp":
    print("TOP_N_PER_TISSUE_TP:", TOP_N_PER_TISSUE_TP)

# Quick coverage of selected candidates.
coverage_selected = (
    selected_configs.groupby(["tissue", "tp"], as_index=False)
    .size()
    .rename(columns={"size": "n_selected_training_candidates"})
)
coverage_selected_path = SAVE_DIR / f"single_omics_selected_config_coverage_{TEST_EXPERIMENT_TAG}.csv"
coverage_selected.to_csv(coverage_selected_path, index=False)
print("selected coverage saved:", coverage_selected_path)

display_cols_selected = [
    "tissue", "tp", "training_rank_within_tissue_tp", "combo",
    "clinical_mode", "feature_selection_mode", "top_k_after_sparse",
    "selected_model_type_from_training", "oof_r2", "mean_valid_r2", "std_valid_r2",
    "mean_train_r2", "oof_mae", "test_selection_basis",
]
display(selected_configs[[c for c in display_cols_selected if c in selected_configs.columns]].head(120))


# ==========================================
# Build jobs from selected training configs
# ==========================================
jobs = []
for _, row in selected_configs.iterrows():
    selected_model = str(row["selected_model_type_from_training"])
    if selected_model not in MODEL_TYPES:
        print(f"[WARN] selected model {selected_model} not in MODEL_TYPES; still passing it directly.")

    jobs.append({
        "feature_mode": FEATURE_MODE_TO_RUN,
        "clinical_mode": str(row["clinical_mode"]),
        "feature_selection_mode": str(row["feature_selection_mode"]),
        "combo": str(row["combo"]),
        "top_k": int(row["top_k_after_sparse"]),
        "tissue": str(row["tissue"]),
        "tp": int(row["tp"]),
        "selected_model_type_from_training": selected_model,
        "train_oof_r2": row.get("oof_r2", np.nan),
        "train_mean_valid_r2": row.get("mean_valid_r2", np.nan),
        "train_std_valid_r2": row.get("std_valid_r2", np.nan),
        "train_mean_train_r2": row.get("mean_train_r2", np.nan),
        "train_oof_mae": row.get("oof_mae", np.nan),
        "training_rank_global": row.get("training_rank_global", np.nan),
        "training_rank_within_tissue_tp": row.get("training_rank_within_tissue_tp", np.nan),
        "training_rank_within_tissue_combo_tp": row.get("training_rank_within_tissue_combo_tp", np.nan),
        "test_selection_basis": row.get("test_selection_basis", "training_only_oof_cv"),
        "holdout_test_usage_note": row.get("holdout_test_usage_note", "evaluation_only"),
    })

print("n_jobs_to_run:", len(jobs))
display(pd.DataFrame(jobs).head(120))


def run_one_single_omics_experiment_safe(job: dict, y_all: pd.Series):
    feature_mode = job["feature_mode"]
    clinical_mode = job["clinical_mode"]
    feature_selection_mode = job["feature_selection_mode"]
    tissue = job["tissue"]
    combo = job["combo"]
    tp = job["tp"]
    top_k = job["top_k"]
    selected_model = job["selected_model_type_from_training"]

    try:
        # Important:
        # Only the model family selected by training-only OOF/CV is allowed here.
        # The test set is not used for model-family selection.
        result = run_one_single_omics_experiment(
            tissue=tissue,
            combo=combo,
            tp=tp,
            y_all=y_all,
            feature_mode=feature_mode,
            candidate_models=[selected_model],
            clinical_mode=clinical_mode,
            feature_selection_mode=feature_selection_mode,
            top_k_after_sparse_override=top_k,
        )
        if result is None:
            raise ValueError("run_one_single_omics_experiment returned None")

        for k in [
            "selected_model_type_from_training",
            "train_oof_r2",
            "train_mean_valid_r2",
            "train_std_valid_r2",
            "train_mean_train_r2",
            "train_oof_mae",
            "training_rank_global",
            "training_rank_within_tissue_tp",
            "training_rank_within_tissue_combo_tp",
            "test_selection_basis",
            "holdout_test_usage_note",
        ]:
            result[k] = job.get(k, np.nan)
        result["train_experiment_tag"] = TRAIN_EXPERIMENT_TAG
        result["test_experiment_tag"] = TEST_EXPERIMENT_TAG
        result["selection_is_training_rank1_within_tissue_tp"] = (
            pd.notna(job.get("training_rank_within_tissue_tp", np.nan))
            and int(job.get("training_rank_within_tissue_tp")) == 1
        )
        return result

    except Exception as e:
        print(
            f"[ERROR] feature_mode={feature_mode}, clinical={clinical_mode}, "
            f"selection={feature_selection_mode}, tissue={tissue}, combo={combo}, "
            f"tp={tp}, top_k={top_k}, selected_model={selected_model} :: {e}"
        )
        return {
            "status": "failed",
            "experiment_tag": EXPERIMENT_TAG,
            "train_experiment_tag": TRAIN_EXPERIMENT_TAG,
            "test_experiment_tag": TEST_EXPERIMENT_TAG,
            "use_feature_weighting": USE_FEATURE_WEIGHTING,
            "feature_weight_score_mode": FEATURE_WEIGHT_SCORE_MODE if USE_FEATURE_WEIGHTING else "none",
            "feature_weight_min": FEATURE_WEIGHT_MIN if USE_FEATURE_WEIGHTING else np.nan,
            "feature_weight_max": FEATURE_WEIGHT_MAX if USE_FEATURE_WEIGHTING else np.nan,
            "mean_feature_weight": np.nan,
            "max_feature_weight": np.nan,
            "n_weighted_features": np.nan,
            "feature_mode": feature_mode,
            "clinical_mode": clinical_mode,
            "feature_selection_mode": feature_selection_mode,
            "tissue": tissue,
            "combo": combo,
            "tp": tp,
            "reason": str(e),
            "best_model_type": selected_model,
            "selected_model_type_from_training": selected_model,
            "n_train": np.nan,
            "n_test": np.nan,
            "train_r2": np.nan,
            "train_mae": np.nan,
            "test_r2": np.nan,
            "test_mae": np.nan,
            "best_score_inner_cv_r2": np.nan,
            "selection_basis": "training_summary_selected_config_then_holdout_test_once",
            "test_selection_basis": job.get("test_selection_basis", "training_only_oof_cv"),
            "holdout_test_usage_note": job.get("holdout_test_usage_note", "evaluation_only"),
            "train_oof_r2": job.get("train_oof_r2", np.nan),
            "train_mean_valid_r2": job.get("train_mean_valid_r2", np.nan),
            "train_std_valid_r2": job.get("train_std_valid_r2", np.nan),
            "train_mean_train_r2": job.get("train_mean_train_r2", np.nan),
            "train_oof_mae": job.get("train_oof_mae", np.nan),
            "training_rank_global": job.get("training_rank_global", np.nan),
            "training_rank_within_tissue_tp": job.get("training_rank_within_tissue_tp", np.nan),
            "training_rank_within_tissue_combo_tp": job.get("training_rank_within_tissue_combo_tp", np.nan),
            "selection_is_training_rank1_within_tissue_tp": False,
            "n_features_input": np.nan,
            "n_delta_features_input": np.nan,
            "n_clinical_input": np.nan,
            "n_features_after_selection": np.nan,
            "top_k_after_sparse": top_k,
        }


rows = Parallel(
    n_jobs=N_JOBS_EXPERIMENT,
    backend="loky",
    verbose=10,
)(
    delayed(run_one_single_omics_experiment_safe)(job, y_all)
    for job in jobs
)

summary_df = pd.DataFrame(rows)

# Ensure key columns exist.
for c in [
    "test_r2", "test_mae", "train_r2", "train_mae",
    "best_score_inner_cv_r2", "train_oof_r2", "train_mean_valid_r2",
    "train_std_valid_r2", "train_mean_train_r2", "train_oof_mae",
    "training_rank_global", "training_rank_within_tissue_tp", "training_rank_within_tissue_combo_tp",
]:
    if c not in summary_df.columns:
        summary_df[c] = np.nan
    summary_df[c] = pd.to_numeric(summary_df[c], errors="coerce")

if "selection_is_training_rank1_within_tissue_tp" not in summary_df.columns:
    summary_df["selection_is_training_rank1_within_tissue_tp"] = False

if len(summary_df) > 0:
    summary_df = summary_df.sort_values(
        by=["status", "tissue", "tp", "training_rank_within_tissue_tp", "combo"],
        ascending=[True, True, True, True, True],
        na_position="last",
    ).reset_index(drop=True)

out_path = SAVE_DIR / f"{FEATURE_MODE_TO_RUN}_holdout_test_summary_{TEST_EXPERIMENT_TAG}.csv"
summary_df.to_csv(out_path, index=False)
print("saved:", out_path)

ok_df = summary_df[summary_df["status"].astype(str).str.lower().eq("ok")].copy()

if len(ok_df) > 0:
    # 1) Leakage-safe final table: training OOF rank-1 per tissue+TP only.
    rank1_df = ok_df[ok_df["training_rank_within_tissue_tp"].eq(1)].copy()
    rank1_df = rank1_df.sort_values(["tissue", "tp"]).reset_index(drop=True)
    rank1_path = SAVE_DIR / f"single_omics_test_training_rank1_by_tissue_tp_for_ef_lf_compare_{TEST_EXPERIMENT_TAG}.csv"
    rank1_df.to_csv(rank1_path, index=False)
    print("saved leakage-safe rank1 final compare table:", rank1_path)

    # 2) Descriptive only: test-best among inspected top-N candidates.
    best_by_test_tissue_tp = (
        ok_df.sort_values("test_r2", ascending=False, na_position="last")
        .groupby(["tissue", "tp"], as_index=False)
        .first()
        .sort_values(["tissue", "tp"])
        .reset_index(drop=True)
    )
    best_by_test_path = SAVE_DIR / f"single_omics_test_descriptive_best_by_test_r2_within_topN_tissue_tp_{TEST_EXPERIMENT_TAG}.csv"
    best_by_test_tissue_tp.to_csv(best_by_test_path, index=False)
    print("saved descriptive test-best table:", best_by_test_path)

    # 3) Descriptive only: test-best per tissue/combo/TP among inspected candidates.
    best_by_test_tissue_combo_tp = (
        ok_df.sort_values("test_r2", ascending=False, na_position="last")
        .groupby(["tissue", "combo", "tp"], as_index=False)
        .first()
        .sort_values(["tissue", "combo", "tp"])
        .reset_index(drop=True)
    )
    best_combo_path = SAVE_DIR / f"single_omics_test_descriptive_best_by_test_r2_within_topN_tissue_combo_tp_{TEST_EXPERIMENT_TAG}.csv"
    best_by_test_tissue_combo_tp.to_csv(best_combo_path, index=False)
    print("saved descriptive test-best combo table:", best_combo_path)

    # 4) Sensitivity summary per tissue+TP.
    sens_rows = []
    for (tissue, tp), g in ok_df.groupby(["tissue", "tp"]):
        g_sorted_train = g.sort_values("training_rank_within_tissue_tp", ascending=True)
        g_sorted_test = g.sort_values("test_r2", ascending=False, na_position="last")
        train_rank1 = g_sorted_train.iloc[0]
        test_best = g_sorted_test.iloc[0]
        sens_rows.append({
            "tissue": tissue,
            "tp": tp,
            "n_tested_topN_candidates": len(g),
            "train_rank1_combo": train_rank1.get("combo"),
            "train_rank1_model": train_rank1.get("selected_model_type_from_training"),
            "train_rank1_fs": train_rank1.get("feature_selection_mode"),
            "train_rank1_top_k": train_rank1.get("top_k_after_sparse"),
            "train_rank1_oof_r2": train_rank1.get("train_oof_r2"),
            "train_rank1_test_r2": train_rank1.get("test_r2"),
            "descriptive_test_best_combo": test_best.get("combo"),
            "descriptive_test_best_model": test_best.get("selected_model_type_from_training"),
            "descriptive_test_best_fs": test_best.get("feature_selection_mode"),
            "descriptive_test_best_top_k": test_best.get("top_k_after_sparse"),
            "descriptive_test_best_training_rank": test_best.get("training_rank_within_tissue_tp"),
            "descriptive_test_best_train_oof_r2": test_best.get("train_oof_r2"),
            "descriptive_test_best_test_r2": test_best.get("test_r2"),
            "test_r2_median_among_topN": g["test_r2"].median(),
            "test_r2_mean_among_topN": g["test_r2"].mean(),
            "test_r2_std_among_topN": g["test_r2"].std(ddof=1) if len(g) > 1 else np.nan,
            "n_positive_test_r2_among_topN": int((g["test_r2"] > 0).sum()),
            "n_negative_test_r2_among_topN": int((g["test_r2"] < 0).sum()),
            "note": "test_best columns are descriptive only; final model must be selected by training OOF/CV, not holdout test R2",
        })
    sensitivity_df = pd.DataFrame(sens_rows).sort_values(["tissue", "tp"]).reset_index(drop=True)
    sensitivity_path = SAVE_DIR / f"single_omics_test_topN_sensitivity_by_tissue_tp_{TEST_EXPERIMENT_TAG}.csv"
    sensitivity_df.to_csv(sensitivity_path, index=False)
    print("saved top-N sensitivity summary:", sensitivity_path)

    # 5) Backward-compatible file name, but make content leakage-safe rank1 by default.
    compare_cols = [
        "status", "tissue", "tp", "combo", "clinical_mode", "feature_selection_mode",
        "top_k_after_sparse", "training_rank_within_tissue_tp",
        "selected_model_type_from_training", "best_model_type",
        "train_oof_r2", "train_mean_valid_r2", "test_r2", "test_mae",
        "train_r2", "best_score_inner_cv_r2", "n_features_after_selection",
        "test_selection_basis", "holdout_test_usage_note",
    ]
    compare_path = SAVE_DIR / f"single_omics_test_for_ef_lf_compare_by_tissue_tp_{TEST_EXPERIMENT_TAG}.csv"
    rank1_df[[c for c in compare_cols if c in rank1_df.columns]].to_csv(compare_path, index=False)
    print("saved EF/LF compare table using training rank1 only:", compare_path)

print("\nHoldout test summary sorted by TEST R² for inspection only.")
print("Do not use this ranking to choose a new final config.")
display_cols = [
    "status", "tissue", "tp", "training_rank_within_tissue_tp", "combo",
    "clinical_mode", "feature_selection_mode", "top_k_after_sparse",
    "selected_model_type_from_training", "best_model_type",
    "train_oof_r2", "train_mean_valid_r2", "test_r2", "test_mae",
    "train_r2", "best_score_inner_cv_r2", "n_features_after_selection",
]
display(summary_df.sort_values("test_r2", ascending=False, na_position="last")[[c for c in display_cols if c in summary_df.columns]].head(120))

if len(ok_df) > 0:
    print("\nLeakage-safe final EF/LF table: training OOF rank-1 per tissue+TP.")
    display(rank1_df[[c for c in display_cols if c in rank1_df.columns]].head(80))

    print("\nTop-N sensitivity summary. Test-best columns are descriptive only.")
    display(sensitivity_df.head(80))

    print("\nDescriptive test-best among inspected top-N candidates. Do not use this as final model selection.")
    display(best_by_test_tissue_tp[[c for c in display_cols if c in best_by_test_tissue_tp.columns]].head(80))


loaded training summary: ../../DifferentCom_data_rebuild/results_single_omics_train_only_single_omics_train_paper_style_v11_rescue_expansion_v9compatible/omics_paper_style_train_summary_single_omics_train_paper_style_v11_rescue_expansion_v9compatible.csv
training summary shape: (148, 39)
training ok rows: 148
selected configs saved: ../../DifferentCom_data_rebuild/testResult/results_single_omics_test_single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible/single_omics_selected_configs_used_for_test_single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible.csv
n_selected_configs: 50
selection mode: top_n_per_tissue_tp
TOP_N_PER_TISSUE_TP: 5
selected coverage saved: ../../DifferentCom_data_rebuild/testResult/results_single_omics_test_single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible/single_omics_selected_config_coverage_single_omics_final_test_top5_sensitivity_from_v11_rescue_expansion_v9compatible.csv


,tissue,tp,training_rank_within_tissue_tp,combo,clinical_mode,feature_selection_mode,top_k_after_sparse,selected_model_type_from_training,oof_r2,mean_valid_r2,std_valid_r2,mean_train_r2,oof_mae,test_selection_basis
0,ser,24,1,C,light,f_regression_topk,40,xgboost,0.271758,0.194197,0.176020,0.733303,15.176796,selected_before_holdout_test_using_training_on...
1,csf,24,1,C,light,corr_topk,30,ridge,0.261029,0.240782,0.225431,0.797261,14.235372,selected_before_holdout_test_using_training_on...
2,csf,24,2,C,light,corr_topk,20,ridge,0.253265,0.316194,0.200290,0.674126,13.439060,selected_before_holdout_test_using_training_on...
3,ser,96,1,A,light,corr_topk,120,svr_linear,0.242782,0.306499,0.302850,0.683036,13.388230,selected_before_holdout_test_using_training_on...
4,csf,96,1,A,light,f_regression_topk,30,ridge,0.236085,0.199354,0.296724,0.501134,13.732454,selected_before_holdout_test_using_training_on...
5,ser,96,2,A,light,f_regression_topk,100,svr_linear,0.234564,0.288056,0.272773,0.634685,13.494215,selected_before_holdout_test_using_training_on...
6,ser,96,3,A,light,corr_topk,100,svr_linear,0.231528,0.283952,0.271167,0.636724,13.546047,selected_before_holdout_test_using_training_on...
7,csf,96,2,A,light,corr_topk,30,ridge,0.226360,0.188781,0.303123,0.517226,13.697032,selected_before_holdout_test_using_training_on...
8,ser,96,4,A,light,f_regression_topk,120,svr_linear,0.217384,0.270970,0.290776,0.660641,13.474596,selected_before_holdout_test_using_training_on...
9,csf,48,1,C,light,corr_topk,30,ridge,0.201590,0.186037,0.241039,0.696185,14.601400,selected_before_holdout_test_using_training_on...


n_jobs_to_run: 50


,feature_mode,clinical_mode,feature_selection_mode,combo,top_k,tissue,tp,selected_model_type_from_training,train_oof_r2,train_mean_valid_r2,train_std_valid_r2,train_mean_train_r2,train_oof_mae,training_rank_global,training_rank_within_tissue_tp,training_rank_within_tissue_combo_tp,test_selection_basis,holdout_test_usage_note
0,omics,light,f_regression_topk,C,40,ser,24,xgboost,0.271758,0.194197,0.176020,0.733303,15.176796,1,1,1,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
1,omics,light,corr_topk,C,30,csf,24,ridge,0.261029,0.240782,0.225431,0.797261,14.235372,2,1,1,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
2,omics,light,corr_topk,C,20,csf,24,ridge,0.253265,0.316194,0.200290,0.674126,13.439060,3,2,2,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
3,omics,light,corr_topk,A,120,ser,96,svr_linear,0.242782,0.306499,0.302850,0.683036,13.388230,4,1,1,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
4,omics,light,f_regression_topk,A,30,csf,96,ridge,0.236085,0.199354,0.296724,0.501134,13.732454,5,1,1,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
5,omics,light,f_regression_topk,A,100,ser,96,svr_linear,0.234564,0.288056,0.272773,0.634685,13.494215,6,2,2,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
6,omics,light,corr_topk,A,100,ser,96,svr_linear,0.231528,0.283952,0.271167,0.636724,13.546047,7,3,3,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
7,omics,light,corr_topk,A,30,csf,96,ridge,0.226360,0.188781,0.303123,0.517226,13.697032,8,2,2,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
8,omics,light,f_regression_topk,A,120,ser,96,svr_linear,0.217384,0.270970,0.290776,0.660641,13.474596,9,4,4,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...
9,omics,light,corr_topk,C,30,csf,48,ridge,0.201590,0.186037,0.241039,0.696185,14.601400,10,1,1,selected_before_holdout_test_using_training_on...,Holdout test is evaluation only. Do not choose...


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


...

,status,tissue,tp,training_rank_within_tissue_tp,combo,clinical_mode,feature_selection_mode,top_k_after_sparse,selected_model_type_from_training,best_model_type,train_oof_r2,train_mean_valid_r2,test_r2,test_mae,train_r2,best_score_inner_cv_r2,n_features_after_selection
0,ok,csf,24,1,C,light,corr_topk,30,ridge,ridge,0.261029,0.240782,-0.301058,15.969808,0.813872,0.159739,30
1,ok,csf,48,1,C,light,corr_topk,30,ridge,ridge,0.201590,0.186037,0.461189,9.556048,0.778636,0.167297,30
2,ok,csf,72,1,A,light,corr_topk,60,gbr,gbr,0.195500,0.126244,0.283918,14.039792,0.411410,0.113992,60
3,ok,csf,96,1,A,light,f_regression_topk,30,ridge,ridge,0.236085,0.199354,0.169784,14.855100,0.491992,0.212344,30
4,ok,csf,120,1,C,light,corr_topk,30,ridge,ridge,0.181183,0.158403,0.131909,13.633584,0.590275,0.096037,30
5,ok,ser,24,1,C,light,f_regression_topk,40,xgboost,xgboost,0.271758,0.194197,-0.598031,17.972773,0.590233,0.176393,40
6,ok,ser,48,1,C,light,f_regression_topk,15,ridge,ridge,0.183097,0.180746,-0.453203,18.736975,0.463340,0.178553,16
7,ok,ser,72,1,C,light,f_regression_topk,5,ridge,ridge,0.157075,0.147370,-0.453203,18.736975,0.463340,0.178553,16
8,ok,ser,96,1,A,light,corr_topk,120,svr_linear,svr_linear,0.242782,0.306499,-0.050829,15.040611,0.543920,0.161328,120
9,ok,ser,120,1,C,light,f_regression_topk,5,ridge,ridge,0.157075,0.147370,-0.453203,18.736975,0.463340,0.178553,16



Top-N sensitivity summary. Test-best columns are descriptive only.


,tissue,tp,n_tested_topN_candidates,train_rank1_combo,train_rank1_model,train_rank1_fs,train_rank1_top_k,train_rank1_oof_r2,train_rank1_test_r2,descriptive_test_best_combo,...,descriptive_test_best_top_k,descriptive_test_best_training_rank,descriptive_test_best_train_oof_r2,descriptive_test_best_test_r2,test_r2_median_among_topN,test_r2_mean_among_topN,test_r2_std_among_topN,n_positive_test_r2_among_topN,n_negative_test_r2_among_topN,note
0,csf,24,5,C,ridge,corr_topk,30,0.261029,-0.301058,A,...,120,5,0.069795,0.175509,-0.063995,-0.093775,0.182799,1,4,test_best columns are descriptive only; final ...
1,csf,48,5,C,ridge,corr_topk,30,0.201590,0.461189,C,...,30,1,0.201590,0.461189,0.161742,0.215837,0.204553,4,1,test_best columns are descriptive only; final ...
2,csf,72,5,A,gbr,corr_topk,60,0.195500,0.283918,A,...,40,3,0.128291,0.369730,0.049997,0.115377,0.210294,4,1,test_best columns are descriptive only; final ...
3,csf,96,5,A,ridge,f_regression_topk,30,0.236085,0.169784,A,...,40,3,0.138209,0.213177,0.169784,0.185127,0.025450,5,0,test_best columns are descriptive only; final ...
4,csf,120,5,C,ridge,corr_topk,30,0.181183,0.131909,C,...,30,3,0.078199,0.152879,0.126571,0.109664,0.039061,5,0,test_best columns are descriptive only; final ...
5,ser,24,5,C,xgboost,f_regression_topk,40,0.271758,-0.598031,C,...,40,1,0.271758,-0.598031,-0.686022,-0.719935,0.112379,0,5,test_best columns are descriptive only; final ...
6,ser,48,5,C,ridge,f_regression_topk,15,0.183097,-0.453203,A,...,40,5,0.082142,0.078223,-0.453203,-0.402895,0.294998,1,4,test_best columns are descriptive only; final ...
7,ser,72,5,C,ridge,f_regression_topk,5,0.157075,-0.453203,A,...,120,4,0.138242,0.228788,0.196684,-0.051213,0.367175,3,2,test_best columns are descriptive only; final ...
8,ser,96,5,A,svr_linear,corr_topk,120,0.242782,-0.050829,A,...,100,2,0.234564,-0.049010,-0.050829,-0.055271,0.012001,0,5,test_best columns are descriptive only; final ...
9,ser,120,5,C,ridge,f_regression_topk,5,0.157075,-0.453203,C,...,20,5,0.045795,-0.286207,-0.453203,-0.414643,0.072662,0,5,test_best columns are descriptive only; final ...



Descriptive test-best among inspected top-N candidates. Do not use this as final model selection.


,status,tissue,tp,training_rank_within_tissue_tp,combo,clinical_mode,feature_selection_mode,top_k_after_sparse,selected_model_type_from_training,best_model_type,train_oof_r2,train_mean_valid_r2,test_r2,test_mae,train_r2,best_score_inner_cv_r2,n_features_after_selection
0,ok,csf,24,5,A,light,f_regression_topk,120,ridge,ridge,0.069795,0.072770,0.175509,14.173174,0.559360,0.006060,120
1,ok,csf,48,1,C,light,corr_topk,30,ridge,ridge,0.201590,0.186037,0.461189,9.556048,0.778636,0.167297,30
2,ok,csf,72,3,A,light,f_regression_topk,40,gbr,gbr,0.128291,0.075298,0.369730,12.881785,0.525343,0.149833,40
3,ok,csf,96,3,A,light,corr_topk,40,ridge,ridge,0.138209,0.071006,0.213177,14.580310,0.513250,0.115292,40
4,ok,csf,120,3,C,light,f_regression_topk,30,ridge,ridge,0.078199,-0.075153,0.152879,13.229077,0.597189,0.125848,30
5,ok,ser,24,1,C,light,f_regression_topk,40,xgboost,xgboost,0.271758,0.194197,-0.598031,17.972773,0.590233,0.176393,40
6,ok,ser,48,5,A,light,corr_topk,40,ridge,ridge,0.082142,0.092378,0.078223,15.196408,0.477922,0.045957,40
7,ok,ser,72,4,A,light,corr_topk,120,svr_linear,svr_linear,0.138242,0.197571,0.228788,12.988432,0.577262,0.103099,120
8,ok,ser,96,3,A,light,corr_topk,100,svr_linear,svr_linear,0.231528,0.283952,-0.049010,14.769857,0.514151,0.178925,100
9,ok,ser,120,5,C,light,f_regression_topk,20,ridge,ridge,0.045795,0.032286,-0.286207,17.214073,0.520281,0.137459,20
